# Path-Robust Risk Controls for Intraday Mean Reversion in NSE Equities
### MScFE 690 · M7 Final Project · Group 17838
Uchenna Splendor Ejike, Israel Olanrewaju Odeajo and Goodness Chibueze Kalu.

Do the selected stop and target improve net returns or downside risk relative to a time-only exit?
The fixed development and validation records show that gross returns did not cover the assumed fees.
This notebook retains the original experiment and evaluates each chronological partition separately.
The default run is validation; the final partition requires reviewed validation and a matching frozen protocol.


## 1. Run instructions

1. Save a copy in Google Drive. Add the instructor's stock and index folders as readable My Drive shortcuts.
2. Set the three folder paths below. Keep `RUN_STAGE="validation"` and `FINAL_ACCESS_APPROVED=False` for the first run.
3. Choose a hosted CPU runtime and **Runtime → Run all**. Read the source audit, accounting checks and results, then save the executed notebook.
4. After reviewing the completed validation, set `READY_TO_FREEZE=True` in the last save cell and rerun **only that cell**. This records the checked protocol without another market replay.
5. Only after the group has approved final access, start a fresh runtime, choose `RUN_STAGE="final"`, set `FINAL_ACCESS_APPROVED=True`, reset `READY_TO_FREEZE=False`, and run all cells once. The exact frozen code, data and settings must match. If a final attempt stops with an error before any final-period result is displayed, keep its access record and notebook and follow the recovery rule in `docs/M7_README.md` step 7; never delete the record.
6. When the group is satisfied with a saved run, tick `PUBLISH_RUN` in the publish cell near the end and rerun only that cell. Until then every run stays in the private save folder.

**Running locally (VS Code or Jupyter):** copy the two instructor folders into the repository `data/` folder and keep the default paths; the first code cell detects that it is not in Colab, uses `data/` and saves runs to `data/m7_runs/`. It shows a setup checklist, and the source check shows every month found before anything is loaded. Install `bsdtar` first (`sudo apt install -y libarchive-tools`).

The notebook streams selected inputs into disk-based SQLite and reads one session at a time. It does not clone a repository or run hidden research scripts. Every run gets a new working folder. Final access is recorded before strategy replay; an existing access record blocks a second automatic run. A failed final attempt must be documented and reviewed before any recovery.


In [ ]:
# Colab form controls: change paths/stage here, not the research rules below.
RUN_STAGE = "validation" # @param ["pilot", "development", "validation", "final"]
FINAL_ACCESS_APPROVED = False # @param {type:"boolean"}
CONNECT_DRIVE = True # @param {type:"boolean"}
CASH_FOLDER = "/content/drive/MyDrive/NSE Cash Stocks Data" # @param {type:"string"}
INDEX_FOLDER = "/content/drive/MyDrive/NSE Index Data" # @param {type:"string"}
SAVE_FOLDER = "/content/drive/MyDrive/WQU_Capstone_17838/m7" # @param {type:"string"}

from pathlib import Path
from datetime import date, datetime, timedelta, timezone
import csv, hashlib, inspect, io, json, math, os, platform, shutil, sqlite3, subprocess, sys, tempfile, time as clock
for variable in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'):
    os.environ[variable] = '1'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

if RUN_STAGE not in {"pilot", "development", "validation", "final"}:
    raise ValueError("Choose pilot, development, validation or final")
if RUN_STAGE == "final" and not FINAL_ACCESS_APPROVED:
    raise RuntimeError("Final test locked: explicit final-access approval is required")
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
SETUP_STEPS = []  # (step, result, detail) shown as a checklist at the end of this cell
# Step 1: where the data live. In Colab the instructor folders are read from Google Drive. Elsewhere,
# if the Drive defaults are unchanged, the repository's own data/ folder is used, so one notebook
# runs in both places without edits. Folder paths never enter the code fingerprint.
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'path_robust').is_dir()), None)
if IN_COLAB:
    if CONNECT_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        SETUP_STEPS.append(('1. Data location', 'Google Drive', 'mounted at /content/drive'))
    else:
        SETUP_STEPS.append(('1. Data location', 'Colab disk', 'CONNECT_DRIVE=False: folder paths used as typed'))
else:
    if CASH_FOLDER.startswith('/content/') or INDEX_FOLDER.startswith('/content/'):
        local_data = REPO_ROOT / 'data' if REPO_ROOT else None
        if not (local_data and (local_data / 'NSE Cash Stocks Data').is_dir() and (local_data / 'NSE Index Data').is_dir()):
            raise RuntimeError(
                'Not running in Colab, and the local data folders were not found.\n'
                '  Option A: open this notebook in Google Colab and keep CONNECT_DRIVE=True.\n'
                '  Option B: copy the instructor folders "NSE Cash Stocks Data" and "NSE Index Data" into the\n'
                '            repository data/ folder (git ignores it), or set CASH_FOLDER and INDEX_FOLDER to them.')
        CASH_FOLDER, INDEX_FOLDER = str(local_data / 'NSE Cash Stocks Data'), str(local_data / 'NSE Index Data')
        SETUP_STEPS.append(('1. Data location', 'Local repository data/', f'{local_data} (Drive defaults replaced)'))
    else:
        SETUP_STEPS.append(('1. Data location', 'Local folders', 'folder paths used as typed; CONNECT_DRIVE ignored outside Colab'))
    if SAVE_FOLDER.startswith('/content/'):
        SAVE_FOLDER = str(REPO_ROOT / 'data' / 'm7_runs' if REPO_ROOT else Path.home() / 'nse_m7_runs')
# Steps 2 and 3: both instructor folders must exist and contain source files.
ROOTS = [Path(CASH_FOLDER), Path(INDEX_FOLDER)]
for step, root in (('2. Stock data folder', ROOTS[0]), ('3. Index data folder', ROOTS[1])):
    if not root.is_dir():
        raise FileNotFoundError(f'{step}: {root} not found. In Colab, add a My Drive shortcut to the instructor '
                                'folder; locally, check the path or copy the folder into data/.')
    found = sum(1 for p in root.rglob('*') if p.is_file() and p.suffix.lower() in ('.rar', '.7z', '.csv'))
    SETUP_STEPS.append((step, 'Found' if found else 'EMPTY', f'{root} ({found} archive or CSV files)'))
# Step 4: a private save folder for this run's exports, freeze and access records.
SAVE_BASE = Path(SAVE_FOLDER)
SAVE_BASE.mkdir(parents=True, exist_ok=True)
SETUP_STEPS.append(('4. Save folder', 'Ready', str(SAVE_BASE)))
WORK_DIR = Path(tempfile.mkdtemp(prefix='nse-research-', dir='/content' if IN_COLAB else None))
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + WORK_DIR.name.rsplit('-',1)[-1]
OUTPUT_DIR = WORK_DIR / 'outputs'
OUTPUT_DIR.mkdir()
# Step 5: an archive reader. Local quality checks may supply a previously audited small database.
# Never a silent fallback.
EXISTING_DB_PATH = globals().get('EXISTING_DB_PATH', None)
if not EXISTING_DB_PATH and not shutil.which('bsdtar') and sys.platform != 'win32':
    if not IN_COLAB:
        raise RuntimeError('5. Archive reader: bsdtar is needed to read the RAR and 7z archives. Install it, then rerun '
                           'this cell:\n  Ubuntu/WSL: sudo apt install -y libarchive-tools\n  macOS: brew install libarchive')
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'libarchive-tools'], check=True)
SETUP_STEPS.append(('5. Archive reader', 'Ready' if (EXISTING_DB_PATH or shutil.which('bsdtar') or sys.platform == 'win32') else 'MISSING',
                    'existing database supplied' if EXISTING_DB_PATH else (shutil.which('bsdtar') or 'Windows tar')))
SETUP_STEPS.append(('6. Stage', RUN_STAGE, 'final access approved' if FINAL_ACCESS_APPROVED else
                    'final test locked' if RUN_STAGE != 'final' else 'final access NOT approved'))
plt.rcParams.update({'figure.dpi':115, 'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.grid':True, 'grid.alpha':0.18, 'font.size':10})
display(Markdown('**Setup checklist**\n\n| Step | Result | Detail |\n|---|---|---|\n'
                 + '\n'.join(f'| {a} | {b} | `{c}` |' for a, b, c in SETUP_STEPS)
                 + '\n\nNext: run the remaining cells in order (Runtime > Run all in Colab; Run All in VS Code). '
                 'The source check below shows every month found before anything is loaded.'))
print('Stage:', RUN_STAGE, '| Runtime:', platform.python_version(), platform.system())
print('Private working directory:', WORK_DIR)


## 2. Research design

The stop-loss impact is related to return dynamics (Kaminski and Lo, 242-44). Our fixed barriers are not shown to be optimal by optimal stopping boundaries based on the models (Leung and Li 15).

| Component | Fixed choice |
|---|---|
| Universe: | Five-stock case study, not selected by profit (RELIANCE, HDFCBANK, INFY, TCS and ITC) |
| Index | `.NSEI`; signal adjusted using the index return with coefficient one without estimated beta or traded hedge |
| Entry | Long at same-clock index adjusted z ≤ −2; no threshold search |
| Exits | Time-only, stop-only, stop plus target, common entries and 15-minute expiry |
| Capital | C₀ = INR 100,000; per episode = INR 10,000; fractional shares, fixed notionals, no borrowing |
| Reference execution: | 15 bp round trip, 1 minute delay, no additional stop penalty and assumed costs |
| Candle paths | O-L-H-C and O-H-L-C |
| Minute labels | Start and end interpretations but supplier convention unconfirmed |
| Primary contrast | Combined minus time: mean daily P&L / C₀, common complete-case dates, reference execution, O-L-H-C |
| Secondary contrasts | Stop minus time; combined minus stop. There is no target only effect identified by this comparison. |


In [ ]:
STUDY_START, STUDY_END = '2021-01-01', '2023-12-31'
CFG = dict(study_title='Path-Robust Risk Controls for Intraday Mean Reversion in NSE Equities',
           start_date=STUDY_START, end_date=STUDY_END,
           universe=['RELIANCE','HDFCBANK','INFY','TCS','ITC'], index_symbol='.NSEI',
           reference_capital_inr=100000.0, allocation_fraction=0.5,
           prior_sessions=20, signal_window_minutes=15, z_threshold=-2.0,
           decision_minutes_after_open=30, decision_interval_minutes=30, holding_minutes=15,
           normal_session_open='09:15', normal_session_close='15:30',
           excluded_nonstandard_sessions=['2021-02-24'], funded_restart_dates={})
SETTINGS = [dict(name='reference',cost_bp=15.0,stop_slippage_bp=0.0,delay_minutes=1),
            dict(name='cost_5',cost_bp=5.0,stop_slippage_bp=0.0,delay_minutes=1),
            dict(name='cost_25',cost_bp=25.0,stop_slippage_bp=0.0,delay_minutes=1),
            dict(name='stop_slippage_5',cost_bp=15.0,stop_slippage_bp=5.0,delay_minutes=1),
            dict(name='stop_slippage_10',cost_bp=15.0,stop_slippage_bp=10.0,delay_minutes=1),
            dict(name='delay_2',cost_bp=15.0,stop_slippage_bp=0.0,delay_minutes=2),
            dict(name='delay_5',cost_bp=15.0,stop_slippage_bp=0.0,delay_minutes=5)]
INFERENCE = dict(confidence=0.95, repetitions=2000, seed=17838,
                 block_lengths=[5,10,20], minimum_block_support=10,
                 short_segment_rule='withhold interval, retain all days in point estimate',
                 method='paired moving blocks separately within contiguous eligible calendar segments')
if RUN_STAGE == 'pilot':
    CFG['end_date'] = '2021-03-31'
display(pd.DataFrame([{'parameter':key, 'value':str(value)} for key,value in CFG.items()]))
display(pd.DataFrame(SETTINGS))


# Synthetic reliability check; it never consumes market returns or changes the strategy.
DIAGNOSTIC_SPEC = dict(replications=200, phis=[0.0, 0.3, 0.7],
    true_mean_bp=0.0, stationary_sd_bp=1.0, process_seed=20261002,
    block_length=5, purpose='Fixed development-mask simulation, not market calibration')


## 3. Calendar and chronological split

Scheduled normal sessions prior to quality exclusions: development will end at floor(0.60n), validation will end at floor(0.80n), and final will hold the remaining sessions. C₀ is the start of each partition. Earlier prices can be used to initialise features, but positions cannot cross days/partitions.

NSE calendars: [2021](https://archives.nseindia.com/content/circulars/CMTR46623.pdf), [2022](https://archives.nseindia.com/content/circulars/CMTR50560.pdf), [2023](https://archives.nseindia.com/content/circulars/CMTR54757.pdf). The [June 2023 amendment cross-reference](https://archives.nseindia.com/content/circulars/CMPT57291.pdf) records the change of Bakri Id to 29th June. The off-calendar rows are reported. Muhurat sessions and Weekends are not part of the normal sessions.

The date of the outage, 24 February 2021 is still set and not available. No simulated trades are made that day, and the exclusion will apply to the next 20 sessions history. This does not estimate outage losses or imply invalidation of all the prices of that day ([SEBI order](https://www.sebi.gov.in/sebi_data/attachdocs/jun-2023/1687270559560.pdf)); it retains the pilot restriction. Documented protocol amendment is required for changes in calendar.


In [ ]:
HOLIDAYS = {
 2021: '01-26 03-11 03-29 04-02 04-14 04-21 05-13 07-21 08-19 09-10 10-15 11-04 11-05 11-19',
 2022: '01-26 03-01 03-18 04-14 04-15 05-03 08-09 08-15 08-31 10-05 10-24 10-26 11-08',
 2023: '01-26 03-07 03-30 04-04 04-07 04-14 05-01 06-29 08-15 09-19 10-02 10-24 11-14 11-27 12-25'}
holiday_dates = {date.fromisoformat(f'{year}-{day}') for year,days in HOLIDAYS.items() for day in days.split()}
ALL_CALENDAR = [d.date() for d in pd.date_range(STUDY_START, STUDY_END)
                if d.weekday()<5 and d.date() not in holiday_dates]
n = len(ALL_CALENDAR)
i60, i80 = math.floor(.60*n), math.floor(.80*n)
PARTITIONS = {'development':ALL_CALENDAR[:i60], 'validation':ALL_CALENDAR[i60:i80],
              'final':ALL_CALENDAR[i80:]}
assert sum(map(len, PARTITIONS.values())) == n
assert max(PARTITIONS['development']) < min(PARTITIONS['validation']) < min(PARTITIONS['final'])
assert date(2023,6,28) in ALL_CALENDAR and date(2023,6,29) not in ALL_CALENDAR
CALENDAR = [d for d in ALL_CALENDAR if CFG['start_date'] <= d.isoformat() <= CFG['end_date']]
EVALUATION_DATES = CALENDAR[CFG['prior_sessions']:] if RUN_STAGE=='pilot' else PARTITIONS[RUN_STAGE]
split_table = pd.DataFrame([dict(stage=k,start=v[0],end=v[-1],scheduled_sessions=len(v))
                           for k,v in PARTITIONS.items()])
display(split_table)
print('Current stage:', EVALUATION_DATES[0], 'to', EVALUATION_DATES[-1],
      '| Final access requires the matching reviewed validation freeze.')
fig, ax = plt.subplots(figsize=(9,1.9))
for j,(name,days) in enumerate(PARTITIONS.items()):
    ax.plot([pd.Timestamp(days[0]),pd.Timestamp(days[-1])],[j,j],lw=11,solid_capstyle='butt',label=name)
ax.set(yticks=range(3), yticklabels=list(PARTITIONS), xlabel='Scheduled calendar date',title='Partitions fixed before exclusions')
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'calendar_split.png'); plt.show()


## Load instructor data

`ROOTS` recognizes the cash-stock and index folders. The inputs supported are monthly archives, yearly index 7z files, or extracted CSVs which have the same monthly folder names as the original, for example `Cash Data January 2021/RELIANCE.csv`. Outer download ZIPs are not supported.

Only the five stocks and `.NSEI` are read. Selected archives are temporarily copied, SHA-256 fingerprinted and streamed into SQLite. If the source is not provided or is unclear, ingestion is halted. The June 2023 supplement is used only through declared amendment `M7-A1-2023-06-23` (cell `research-011`), behind its quality gates.

Validation rules

For dates, use explicit ones starting with `MM/DD/YYYY` or `DD-MM-YYYY` dates; cash switches to the latter in August of 2023. Finite positive OHLC values with consistent ranges should be used. Within-month duplicate or backwards timestamp and malformed rows do not work. Only when the exact timestamp/OHLCV match occurs with the proper month's validated data then the data for August 22 is excluded, which was copied in September. Exclusions are counted. September 22 is still missing and dates are never relabelled. Forward fill and/or silent adjustment or inferred calendar is not used. Raw minute labels are kept; verification of timezone and label conventions has not been done. The optional 9th column is not used and is recorded as unused.

The data is specified as: Outputs: `DB_PATH`, `DB_READONLY_URI`, `DATA_AUDIT`, `COVERAGE_ROWS`; table `bars(symbol, stamp, o, h, l, c, volume, source_id)`. `EXISTING_DB_PATH` is a local check labelled, not checking raw Drive inputs.


In [ ]:
import calendar, csv, hashlib, io, json, math, re, shutil, sqlite3, subprocess, sys, tempfile
from collections import defaultdict
from contextlib import closing, contextmanager
from datetime import date, datetime
from pathlib import Path

def month_keys(start, end):
    first, last = date.fromisoformat(start), date.fromisoformat(end)
    if first > last:
        raise ValueError('start_date must be on or before end_date')
    year, month = first.year, first.month
    result = []
    while (year, month) <= (last.year, last.month):
        result.append(f'{year:04d}-{month:02d}')
        year, month = (year + 1, 1) if month == 12 else (year, month + 1)
    return result

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def archive_command(*args):
    executable = shutil.which('bsdtar') or (shutil.which('tar') if sys.platform == 'win32' else None)
    if executable is None:
        raise RuntimeError('Install libarchive-tools in Colab to provide bsdtar.')
    return [executable, *map(str, args)]

@contextmanager
def archive_lines(path, member=None):
    command = archive_command('-tf', path) if member is None else archive_command('-xOf', path, member)
    with tempfile.TemporaryFile(mode='w+t') as errors:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=errors,
                                   text=True, encoding='utf-8-sig')
        try:
            yield process.stdout
            process.stdout.close()
            if process.wait():
                errors.seek(0)
                raise RuntimeError(errors.read(2000))
        finally:
            if process.poll() is None:
                process.kill()
                process.wait()
            process.stdout.close()

def member_named(names, basename):
    matches = [name for name in names if name.replace('\\', '/').rsplit('/', 1)[-1] == basename]
    if len(matches) != 1:
        raise ValueError(f'Expected one {basename}; found {len(matches)}')
    return matches[0]

def archive_members(path):
    with archive_lines(path) as lines:
        return [line.rstrip('\r\n') for line in lines]

def vendor_rows(lines, symbol, month=None):
    reader = csv.reader(lines, strict=True)
    fields = ('<ticker>', '<date>', '<time>', '<open>', '<high>', '<low>', '<close>', '<volume>')
    header = tuple(item.strip().lstrip('\ufeff') for item in next(reader, []))
    if header != fields and not (len(header) == 9 and header[:8] == fields and header[8] in ('<o/i>', '')):
        raise ValueError('Unsupported/absent header; headerless supplements are not accepted')
    previous = None
    for row in reader:
        try:
            if len(row) != len(header) or row[0].strip() != symbol:
                raise ValueError('Wrong column count or ticker')
            day, clock = row[1].strip(), row[2].strip()
            if re.fullmatch(r'[0-9]{2}/[0-9]{2}/[0-9]{4}', day):
                date_format = '%m/%d/%Y'
            elif re.fullmatch(r'[0-9]{2}-[0-9]{2}-[0-9]{4}', day):
                date_format = '%d-%m-%Y'
            else:
                raise ValueError(f'Expected MM/DD/YYYY or DD-MM-YYYY, received {day!r}')
            if not re.fullmatch(r'[0-9]{2}:[0-9]{2}:00', clock):
                raise ValueError(f'Expected minute-boundary HH:MM:00, received {clock!r}')
            stamp = datetime.strptime(day + ' ' + clock, date_format + ' %H:%M:%S')
            o, h, l, c, volume = map(float, row[3:8])
            if not all(math.isfinite(x) and x > 0 for x in (o, h, l, c)) or not l <= min(o, c) <= max(o, c) <= h:
                raise ValueError('Invalid OHLC prices/range')
            if not math.isfinite(volume) or volume < 0:
                raise ValueError('Invalid volume')
            if month is None or stamp.strftime('%Y-%m') == month:
                if previous is not None and stamp <= previous:
                    raise ValueError('Duplicate or out-of-order timestamp')
                previous = stamp
        except (ValueError, OverflowError) as error:
            raise ValueError(f'{symbol}, CSV line {reader.line_num}: {error}') from error
        yield stamp.isoformat(), o, h, l, c, volume


In [ ]:
def discover_sources(roots, cfg):
    roots = list(dict.fromkeys(Path(root).resolve() for root in roots))
    for root in roots:
        if not root.is_dir():
            raise FileNotFoundError(f'Check the mounted Drive folder: {root}')
    files = sorted({path.resolve() for root in roots for path in root.rglob('*')
                    if path.is_file() and path.suffix.lower() in ('.csv', '.rar', '.7z')})
    by_name = defaultdict(list)
    for path in files:
        by_name[path.name].append(path)
    plan, missing = [], []
    for ym in month_keys(cfg['start_date'], cfg['end_date']):
        year, number = map(int, ym.split('-'))
        month = calendar.month_name[number]
        for stem, symbols in ((f'Cash Data {month} {year}', cfg['universe']),
                              (f'NSE Indices-{month} {year}', [cfg['index_symbol']])):
            direct = by_name[stem + '.rar']
            yearly = by_name[f'NSE-Index-{year}.7z'] if symbols == [cfg['index_symbol']] else []
            extracted = {symbol: [p for p in by_name[symbol + '.csv'] if stem in [q.name for q in p.parents]]
                         for symbol in symbols}
            has_csv = any(extracted.values())
            choices = len(direct) + len(yearly) + int(has_csv)
            if choices > 1 or any(len(paths) > 1 for paths in extracted.values()):
                raise ValueError(f'Ambiguous copies of {stem}; choose one source root/layout')
            if has_csv:
                for symbol, paths in extracted.items():
                    if not paths:
                        missing.append(f'{ym}: {symbol}.csv in {stem}')
                    else:
                        plan.append(dict(kind='csv', path=paths[0], month=ym, symbols=[symbol]))
            elif direct or yearly:
                plan.append(dict(kind='rar' if direct else 'year', path=(direct or yearly)[0],
                                 month=ym, symbols=list(symbols), member=stem + '.rar'))
            else:
                missing.append(stem + '.rar or extracted CSVs')
    supplements = [str(p) for p in files if 'missing' in p.name.lower()]
    return plan, missing, supplements

# Synthetic checks: these contain no market data and never open a Drive file.
header = '<ticker>,<date>,<time>,<open>,<high>,<low>,<close>,<volume>\n'
row = 'SYNTHETIC,03/04/2021,09:15:00,100,101,99,100.5,250\n'
assert list(vendor_rows(io.StringIO(header + row), 'SYNTHETIC'))[0][0] == '2021-03-04T09:15:00'
assert len(list(vendor_rows(io.StringIO(chr(65279) + header.replace(chr(10), chr(13)+chr(10)) + row), 'SYNTHETIC'))) == 1
for bad in (row + row, row.replace('100,101,99', '100,99,98'), row.replace('09:15:00', '09:15:01')):
    try:
        list(vendor_rows(io.StringIO(header + bad), 'SYNTHETIC'))
    except ValueError:
        pass
    else:
        raise AssertionError('Malformed synthetic data was accepted')
with tempfile.TemporaryDirectory() as demo:
    folder = Path(demo)
    (folder / 'Cash Data January 2021.rar').touch()
    (folder / 'NSE-Index-2021.7z').touch()
    cfg_demo = dict(start_date='2021-01-01', end_date='2021-01-31', universe=['SYNTHETIC'], index_symbol='.NSEI')
    plan_demo, missing_demo, _ = discover_sources([folder], cfg_demo)
    assert len(plan_demo) == 2 and not missing_demo
    (folder / 'Cash Data January 2021').mkdir()
    (folder / 'Cash Data January 2021' / 'SYNTHETIC.csv').touch()
    try:
        discover_sources([folder], cfg_demo)
    except ValueError:
        pass
    else:
        raise AssertionError('Ambiguous synthetic sources were accepted')
print('Synthetic parser and source-discovery checks passed.')


In [ ]:
def build_database(plan, cfg, work_dir):
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    # A fresh attempt preserves any partial database from a failed read.
    database = Path(tempfile.mkdtemp(prefix='input-', dir=work_dir)) / 'bars.sqlite'
    con = sqlite3.connect(database)
    con.execute('PRAGMA cache_size=-2048')
    con.execute('PRAGMA temp_store=FILE')
    con.execute('CREATE TABLE bars(symbol TEXT, stamp TEXT, o REAL,h REAL,l REAL,c REAL,volume REAL,source_id INTEGER,PRIMARY KEY(symbol,stamp)) WITHOUT ROWID')
    manifest = []

    def ingest(lines, symbol, item, provenance):
        source_id, rows, retained, first, last = len(manifest) + 1, 0, 0, None, None
        digest = hashlib.sha256()
        duplicates = 0
        for stamp, o, h, l, c, volume in vendor_rows(lines, symbol, month=item['month']):
            rows += 1
            digest.update(f'{symbol},{stamp},{o!r},{h!r},{l!r},{c!r},{volume!r}\n'.encode())
            if stamp[:7] != item['month']:
                known = con.execute('SELECT o,h,l,c,volume FROM bars WHERE symbol=? AND stamp=?',
                                    (symbol, stamp)).fetchone()
                if known != (o, h, l, c, volume):
                    raise ValueError(f'Unmatched/conflicting out-of-month row: {symbol}, {stamp}')
                duplicates += 1
                continue  # Exact copies of an already validated monthly source; never relabel dates.
            first, last = first or stamp, stamp
            # Boundary months are fully validated; dates outside the requested range are counted.
            if cfg['start_date'] <= stamp[:10] <= cfg['end_date']:
                con.execute('INSERT INTO bars VALUES(?,?,?,?,?,?,?,?)', (symbol, stamp, o, h, l, c, volume, source_id))
                retained += 1
        if not rows or not retained:
            raise ValueError(f'Empty requested sample for {symbol}, {item["month"]}')
        manifest.append(dict(source_id=source_id, symbol=symbol, month=item['month'], rows=rows,
                             retained_rows=retained, outside_requested_dates=rows-retained-duplicates,
                             excluded_exact_out_of_month_duplicates=duplicates,
                             first_label=first, last_label=last, parsed_ohlcv_sha256=digest.hexdigest(), **provenance))
        con.commit()

    grouped = defaultdict(list)
    for item in plan:
        grouped[item['path']].append(item)
    try:
        for source, items in grouped.items():
            if items[0]['kind'] == 'csv':
                provenance = dict(source=str(source), csv_sha256=file_hash(source), optional_ninth_field='not used')
                with source.open(encoding='utf-8-sig', newline='') as lines:
                    ingest(lines, items[0]['symbols'][0], items[0], provenance)
            else:
                with tempfile.TemporaryDirectory(dir=work_dir) as staging:
                    staged = Path(staging) / source.name
                    shutil.copyfile(source, staged)
                    source_hash, names = file_hash(staged), archive_members(staged)
                    for item in items:
                        monthly = staged
                        provenance = dict(source=str(source), archive_sha256=source_hash, optional_ninth_field='not used')
                        if item['kind'] == 'year':
                            nested = member_named(names, item['member'])
                            monthly = Path(staging) / item['member']
                            with monthly.open('wb') as output:
                                subprocess.run(archive_command('-xOf', staged, nested), stdout=output, check=True)
                            provenance.update(year_member=nested, monthly_archive_sha256=file_hash(monthly))
                        monthly_names = names if monthly == staged else archive_members(monthly)
                        for symbol in item['symbols']:
                            member = member_named(monthly_names, symbol + '.csv')
                            with archive_lines(monthly, member) as lines:
                                ingest(lines, symbol, item, {**provenance, 'member': member})
                        if monthly != staged:
                            monthly.unlink()  # Only the temporary copy made immediately above.
            print('Validated:', source.name)
        audit = dict(mode='raw instructor inputs', raw_inputs_reverified=True, input_files=manifest,
                     requested_start=cfg['start_date'], requested_end=cfg['end_date'],
                     assumptions='Raw naive minute labels; timezone and interval convention unverified; no filling or adjustment.')
        (work_dir / 'input_audit.json').write_text(json.dumps(audit, indent=2) + '\n')
        return database, audit
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()


### Declared source amendment and known final-period coverage (pre-freeze)

The June 2023 monthly cash file lacks 14:16-14:57 on 23 June 2023 for all five stocks, and its 14:15 and 14:58 bars are partial. The instructor supplement `Cash - June 2023- Missing Data.7z` is a complete, headerless re-delivery of that session. Amendment `M7-A1-2023-06-23` replaces the five stocks' bars for that one date with the supplement, merging its single pre-open auction row into 09:15 as the monthly files do. The code stops if the supplement is missing, covers another date, misses any session minute, or disagrees with the monthly file on more than 5% of shared minutes or by more than 50 bp in any price. No other date, file or the index is changed.

The NIFTY 50 file also lacks 14:16-14:57 on 23 June 2023 (333 of 375 session minutes, checked against the raw file), and no index supplement exists. The 14:45 decision on that date therefore has no index return, so under the unchanged complete-day rule 23 June and the 20 following sessions are incomplete. The amendment does not change which sessions are evaluable; it keeps the funded accounts valued from delivered prices if a 14:15 signal fires that day.

Coverage metadata shows four more final-period gaps, none amended: RELIANCE lacks the opening 44 minutes on 20 July 2023, all stocks are missing on 22 September 2023, and 2 or 3 minutes are missing for every stock on 12 September and 26 October 2023. Under the unchanged rules about 88 of the 148 final sessions are expected to be evaluable, or 60 if the September and October minutes fall in signal windows. The final run's sample accounting reports the actual numbers. All of this was declared from coverage metadata before the freeze and before any final-test outcome was computed; no rule, date or threshold is changed in response. The full record is `config/m7_source_amendment.md`.

In [ ]:
from contextlib import closing
from datetime import datetime, timedelta

# ---- M7 pre-freeze source amendment: declared before any final-test outcome was computed ----
SOURCE_AMENDMENTS = [dict(
    amendment_id='M7-A1-2023-06-23',
    trade_date='2023-06-23',
    archive_name='Cash - June 2023- Missing Data.7z',
    layout='headerless: YYYYMMDD,HH:MM,open,high,low,close,volume,extra',
    rule=('For the study stocks on this date only, delete every monthly-file bar and insert the supplement '
          'session. Its single flat pre-open row is merged into the session-open bar as the monthly files do: '
          'open = pre-open price, high/low over both rows, close = open-bar close, volumes summed.'),
    reason=('The June 2023 monthly file lacks 14:16-14:57 for all five stocks and its 14:15 and 14:58 bars '
            'are partial; the instructor supplement is a complete re-delivery of the session.'),
    min_identical_share=0.95, max_shared_price_gap_bp=50.0)]  # a list, so the frozen JSON copy compares equal


def supplement_rows(lines, symbol, trade_date):
    """Parse the headerless one-day supplement layout; any unexpected field stops the run."""
    reader = csv.reader(lines, strict=True)
    day_token = trade_date.replace('-', '')
    rows, previous = [], None
    for row in reader:
        if not any(field.strip() for field in row):
            continue
        try:
            if len(row) != 8:
                raise ValueError('Expected 8 fields: date,time,open,high,low,close,volume,extra')
            if row[0].strip().lstrip('﻿') != day_token:
                raise ValueError(f'Unexpected date {row[0]!r}')
            clock = row[1].strip()
            if not re.fullmatch(r'[0-9]{2}:[0-9]{2}', clock):
                raise ValueError(f'Expected HH:MM, received {clock!r}')
            stamp = datetime.strptime(trade_date + ' ' + clock, '%Y-%m-%d %H:%M')
            o, h, l, c, volume = map(float, row[2:7])
            if not all(math.isfinite(x) and x > 0 for x in (o, h, l, c)) or not l <= min(o, c) <= max(o, c) <= h:
                raise ValueError('Invalid OHLC prices/range')
            if not math.isfinite(volume) or volume < 0:
                raise ValueError('Invalid volume')
            if previous is not None and stamp <= previous:
                raise ValueError('Duplicate or out-of-order timestamp')
            previous = stamp
        except (ValueError, OverflowError) as error:
            raise ValueError(f'{symbol} supplement, CSV line {reader.line_num}: {error}') from error
        rows.append((stamp.isoformat(), o, h, l, c, volume))
    if not rows:
        raise ValueError(f'{symbol} supplement is empty')
    return rows


def fold_preopen(rows, session_open):
    """Merge the single pre-open auction row into the session-open bar."""
    open_label = rows[0][0][:11] + session_open + ':00'
    early = [r for r in rows if r[0] < open_label]
    rest = [r for r in rows if r[0] >= open_label]
    if len(early) != 1 or not rest or rest[0][0] != open_label:
        raise ValueError('Expected exactly one pre-open row and a bar at the session open')
    pre, first = early[0], rest[0]
    if not pre[1] == pre[2] == pre[3] == pre[4]:
        raise ValueError('Pre-open row is not a single auction price')
    merged = (first[0], pre[1], max(pre[2], first[2]), min(pre[3], first[3]), first[4], pre[5] + first[5])
    return [merged] + rest[1:]


def apply_source_amendments(db_path, roots, cfg, work_dir, amendments):
    """Merge (insert supplement), prune (delete original day) and quality-check each declared amendment."""
    audit = []
    for amendment in amendments:
        day = amendment['trade_date']
        if not cfg['start_date'] <= day <= cfg['end_date']:
            audit.append(dict(amendment_id=amendment['amendment_id'], status='outside requested dates'))
            continue
        matches = sorted({p.resolve() for root in roots for p in Path(root).rglob(amendment['archive_name']) if p.is_file()})
        if len(matches) != 1:
            raise FileNotFoundError(f"Expected one {amendment['archive_name']}; found {len(matches)}")
        start = datetime.strptime(day + ' ' + cfg['normal_session_open'], '%Y-%m-%d %H:%M')
        end = datetime.strptime(day + ' ' + cfg['normal_session_close'], '%Y-%m-%d %H:%M')
        session = {(start + timedelta(minutes=m)).isoformat() for m in range(int((end - start).total_seconds() // 60))}
        records = []
        with tempfile.TemporaryDirectory(dir=work_dir) as staging:
            staged = Path(staging) / matches[0].name
            shutil.copyfile(matches[0], staged)
            archive_sha256, names = file_hash(staged), archive_members(staged)
            with closing(sqlite3.connect(db_path)) as con:
                source_id = (con.execute('SELECT COALESCE(MAX(source_id), 0) FROM bars').fetchone()[0] or 0) + 1
                for symbol in cfg['universe']:
                    member = member_named(names, symbol + '.csv')
                    with archive_lines(staged, member) as lines:
                        new_rows = fold_preopen(supplement_rows(lines, symbol, day), cfg['normal_session_open'])
                    new = {r[0]: r[1:] for r in new_rows}
                    if any(stamp[:10] != day for stamp in new):
                        raise ValueError(f'{symbol} supplement contains another date')
                    if session - set(new):
                        raise ValueError(f'{symbol} supplement does not cover the full session')
                    old = {r[0]: tuple(r[1:]) for r in con.execute(
                        'SELECT stamp,o,h,l,c,volume FROM bars WHERE symbol=? AND substr(stamp,1,10)=?', (symbol, day))}
                    shared = sorted(set(old) & set(new) & session)
                    identical = sum(old[t] == new[t] for t in shared)
                    gap_bp = max((abs(a - b) / a * 1e4 for t in shared for a, b in zip(old[t][:4], new[t][:4])), default=0.0)
                    if not shared or identical / len(shared) < amendment['min_identical_share']:
                        raise ValueError(f'{symbol}: supplement agrees on too few shared minutes ({identical}/{len(shared)})')
                    if gap_bp > amendment['max_shared_price_gap_bp']:
                        raise ValueError(f'{symbol}: shared-minute price gap {gap_bp:.1f} bp exceeds the declared limit')
                    digest = hashlib.sha256()
                    for stamp, o, h, l, c, volume in new_rows:
                        digest.update(f'{symbol},{stamp},{o!r},{h!r},{l!r},{c!r},{volume!r}\n'.encode())
                    con.execute('DELETE FROM bars WHERE symbol=? AND substr(stamp,1,10)=?', (symbol, day))
                    con.executemany('INSERT INTO bars VALUES(?,?,?,?,?,?,?,?)',
                                    [(symbol, stamp, *values, source_id) for stamp, *values in new_rows])
                    records.append(dict(symbol=symbol, member=member, original_rows_removed=len(old),
                                        supplement_rows_inserted=len(new_rows),
                                        session_minutes_missing_before=len(session - set(old)),
                                        session_minutes_filled=len((session - set(old)) & set(new)),
                                        shared_session_minutes=len(shared), identical_shared_minutes=identical,
                                        max_shared_price_gap_bp=round(gap_bp, 3),
                                        parsed_ohlcv_sha256=digest.hexdigest()))
                index_minutes = {r[0] for r in con.execute(
                    'SELECT stamp FROM bars WHERE symbol=? AND substr(stamp,1,10)=?', (cfg['index_symbol'], day))}
                con.commit()
        audit.append(dict(amendment_id=amendment['amendment_id'], status='applied', trade_date=day,
                          archive=matches[0].name, archive_sha256=archive_sha256, source_id=source_id,
                          rule=amendment['rule'], reason=amendment['reason'], symbols=records,
                          index_session_minutes_missing=len(session - index_minutes)))
    return audit


def source_status_markdown(plan, missing, supplements, cfg, amendments):
    """Month-by-month table of the source the loader will use, shown before ingestion."""
    labels = {'rar': 'RAR', 'year': '7z', 'csv': 'CSV'}
    status = {}
    for item in plan:
        group = 'Index' if item['symbols'] == [cfg['index_symbol']] else 'Stocks'
        status[(group, item['month'])] = labels[item['kind']]
    months = {name: number for number, name in enumerate(calendar.month_name) if name}
    for text in missing:
        found = re.search(r'(' + '|'.join(months) + r') ([0-9]{4})', text)
        if found:
            ym = f'{int(found.group(2)):04d}-{months[found.group(1)]:02d}'
            status[('Stocks' if 'Cash Data' in text else 'Index', ym)] = 'MISSING'
    keys = month_keys(cfg['start_date'], cfg['end_date'])
    years = sorted({k[:4] for k in keys})
    lines = ['**Source check** (one cell per month: RAR, 7z or CSV is the file that will be read)', '',
             '| Data | Year | ' + ' | '.join(calendar.month_abbr[m] for m in range(1, 13)) + ' |',
             '|---|---|' + '---|' * 12]
    for group in ('Stocks', 'Index'):
        for year in years:
            cells = [status.get((group, f'{year}-{m:02d}'), '') if f'{year}-{m:02d}' in keys else ''
                     for m in range(1, 13)]
            lines.append(f'| {group} | {year} | ' + ' | '.join(cells) + ' |')
    planned = {a['archive_name']: a for a in amendments if cfg['start_date'] <= a['trade_date'] <= cfg['end_date']}
    for path in supplements:
        name = Path(path).name
        use = (f"used by declared amendment {planned[name]['amendment_id']} ({planned[name]['trade_date']} only)"
               if name in planned else 'not used (no declared amendment)')
        lines.append(f'\nSupplement `{name}`: {use}')
    gaps = sum(v == 'MISSING' for v in status.values())
    lines.append(f'\n{"All requested months found." if not gaps else f"{gaps} month(s) MISSING: add the files, then rerun this cell."}')
    return '\n'.join(lines)


existing = globals().get('EXISTING_DB_PATH')
if existing:
    DB_PATH = Path(existing).resolve()
    if not DB_PATH.is_file():
        raise FileNotFoundError(DB_PATH)
    DATA_AUDIT = dict(mode='EXISTING DATABASE: LOCAL SMOKE TEST ONLY', raw_inputs_reverified=False,
                      database_sha256=file_hash(DB_PATH), requested_start=CFG['start_date'],
                      requested_end=CFG['end_date'], assumptions='Existing raw-label database; source inputs not reverified here.')
    DATA_AUDIT['source_amendments'] = 'not applied: existing database smoke test'
    print(DATA_AUDIT['mode'])
else:
    try:
        SOURCE_PLAN, MISSING_SOURCES, EXCLUDED_SUPPLEMENTS = discover_sources(ROOTS, CFG)
    except ValueError as error:
        display(Markdown(f'**Source layout problem:** {error}\n\nKeep one layout per month: either the archive '
                         '(.rar or .7z) or an extracted folder of CSV files, not both. Move one copy out of the data folders.'))
        raise
    display(Markdown(source_status_markdown(SOURCE_PLAN, MISSING_SOURCES, EXCLUDED_SUPPLEMENTS, CFG, SOURCE_AMENDMENTS)))
    print('Requested months:', len(month_keys(CFG['start_date'], CFG['end_date'])))
    print('Selected source entries:', len(SOURCE_PLAN))
    print('Supplements found (use shown in the source check):', [Path(p).name for p in EXCLUDED_SUPPLEMENTS])
    if MISSING_SOURCES:
        print('Missing requested sources:\n' + '\n'.join(MISSING_SOURCES))
        raise FileNotFoundError('Resolve missing sources or explicitly revise the requested scope before ingestion.')
    DB_PATH, DATA_AUDIT = build_database(SOURCE_PLAN, CFG, WORK_DIR)
    SOURCE_AMENDMENT_AUDIT = apply_source_amendments(DB_PATH, ROOTS, CFG, WORK_DIR, SOURCE_AMENDMENTS)
    DATA_AUDIT['source_amendments'] = SOURCE_AMENDMENT_AUDIT
    used = {a['archive'] for a in SOURCE_AMENDMENT_AUDIT if a['status'] == 'applied'}
    DATA_AUDIT['excluded_supplements'] = [p for p in EXCLUDED_SUPPLEMENTS if Path(p).name not in used]
    for a in SOURCE_AMENDMENT_AUDIT:
        print('Source amendment', a['amendment_id'] + ':', a['status'])
        for r in a.get('symbols', []):
            print(f"  {r['symbol']}: removed {r['original_rows_removed']}, inserted {r['supplement_rows_inserted']}, "
                  f"filled {r['session_minutes_filled']} missing minutes, identical {r['identical_shared_minutes']}/{r['shared_session_minutes']} shared, "
                  f"max price gap {r['max_shared_price_gap_bp']} bp")
        if a['status'] == 'applied':
            print('  index session minutes missing on that date:', a['index_session_minutes_missing'])
    print('Imported rows:', sum(x['retained_rows'] for x in DATA_AUDIT['input_files']),
          '| Verified out-of-month copies excluded:', sum(x['excluded_exact_out_of_month_duplicates'] for x in DATA_AUDIT['input_files']))
    (Path(WORK_DIR) / 'input_audit.json').write_text(json.dumps(DATA_AUDIT, indent=2) + '\n')
DB_READONLY_URI = DB_PATH.resolve().as_uri() + '?mode=ro'


### Coverage counts

All requested symbol-months need to be available. Counts individual raw labels outside time window 09:15-15:29 and counts labels that are within the time window separately, 375 labels alone do not verify the timestamp convention. Missing minutes remain missing. Aggregate exports include no detailed audit paths or raw prices.


In [ ]:
with closing(sqlite3.connect(DB_READONLY_URI, uri=True)) as con:
    required_columns = ['symbol', 'stamp', 'o', 'h', 'l', 'c', 'volume', 'source_id']
    if [row[1] for row in con.execute('PRAGMA table_info(bars)')] != required_columns:
        raise ValueError('Unexpected database schema')
    scope = (CFG['start_date'], CFG['end_date'])
    coverage = con.execute("""
        SELECT symbol, substr(stamp,1,7), COUNT(*), COUNT(DISTINCT substr(stamp,1,10)), MIN(stamp), MAX(stamp)
        FROM bars WHERE substr(stamp,1,10) BETWEEN ? AND ?
        GROUP BY symbol, substr(stamp,1,7) ORDER BY 2,1""", scope).fetchall()
    observed = {(row[0], row[1]) for row in coverage}
    expected = {(symbol, ym) for symbol in [*CFG['universe'], CFG['index_symbol']]
                for ym in month_keys(*scope)}
    if expected - observed:
        raise ValueError(f'Database does not cover requested symbol-months: {sorted(expected-observed)}')
    if observed - expected:
        raise ValueError(f'Unexpected symbols in database scope: {sorted(observed-expected)}')
    daily = con.execute("""
        SELECT symbol, substr(stamp,1,10), COUNT(*),
               SUM(substr(stamp,12)>='09:15:00' AND substr(stamp,12)<'15:30:00')
        FROM bars WHERE substr(stamp,1,10) BETWEEN ? AND ?
        GROUP BY symbol, substr(stamp,1,10) ORDER BY 2,1""", scope).fetchall()
COVERAGE_ROWS = [dict(zip(('symbol', 'month', 'rows', 'observed_days', 'first_label', 'last_label'), row)) for row in coverage]
RAW_LABEL_DAY_COUNTS = [dict(symbol=symbol, date=day, rows=rows, raw_0915_1529_labels=regular,
                             outside_window_labels=rows-regular) for symbol, day, rows, regular in daily]
DATA_AUDIT['coverage'] = COVERAGE_ROWS
DATA_AUDIT['raw_label_day_counts'] = RAW_LABEL_DAY_COUNTS
if not globals().get('EXISTING_DB_PATH'):
    (Path(WORK_DIR) / 'input_audit.json').write_text(json.dumps(DATA_AUDIT, indent=2) + '\n')
print('Symbol       Month      Rows  Observed days')
for row in COVERAGE_ROWS:
    print(f'{row["symbol"]:12} {row["month"]} {row["rows"]:7,d} {row["observed_days"]:14}')
print('Symbol-days with other than 375 raw labels in 09:15–15:29:', sum(row['raw_0915_1529_labels'] != 375 for row in RAW_LABEL_DAY_COUNTS))
print('No price, return, holdout result or inferred trading calendar was displayed.')


### Coverage heatmap

Compare both timestamp interpretations to 375 normal-session labels. If you miss a day, that day is counted as 0 observations, but not 0 strategy returns. Volume does not represent spread, depth or queue priority.

Adjustments for corporate actions and tick/lot conventions are not verified. These data limitations do not go away during intraday exposure.


In [ ]:
coverage_table = pd.DataFrame(COVERAGE_ROWS)
display(coverage_table)
symbols = CFG['universe'] + [CFG['index_symbol']]
coverage_records = []
with sqlite3.connect(DB_READONLY_URI, uri=True) as con:
    for label, lo, hi in [('start','09:15:00','15:29:00'),('end','09:16:00','15:30:00')]:
        rows = con.execute("""
          SELECT symbol,substr(stamp,1,10),count(*) FROM bars
          WHERE substr(stamp,12) BETWEEN ? AND ? GROUP BY symbol,substr(stamp,1,10)
        """, (lo,hi)).fetchall()
        count_map = {(s,d):v for s,d,v in rows}
        for d in CALENDAR:
            for s in symbols:
                coverage_records.append(dict(timestamp_label=label,date=d.isoformat(),symbol=s,
                                             observed_minutes=count_map.get((s,d.isoformat()),0)))
coverage_daily = pd.DataFrame(coverage_records)
off_calendar = sorted({r['date'] for r in RAW_LABEL_DAY_COUNTS} - {d.isoformat() for d in CALENDAR})
print('Raw dates outside the declared normal-session calendar:', off_calendar)
fig, axes = plt.subplots(2,1,figsize=(12,4.8),sharex=True)
for ax,label in zip(axes,['start','end']):
    grid = coverage_daily.query('timestamp_label == @label').pivot(index='symbol',columns='date',values='observed_minutes').reindex(symbols)
    img = ax.imshow(grid.values,aspect='auto',vmin=0,vmax=375,cmap='viridis')
    ticks = np.linspace(0,len(grid.columns)-1,min(7,len(grid.columns))).astype(int)
    ax.set(yticks=range(len(symbols)),yticklabels=symbols,title=f'{label}-label assumption: normal-session coverage',
           xticks=ticks,xticklabels=grid.columns[ticks])
    fig.colorbar(img,ax=ax,label='Minutes observed')
axes[-1].set_xlabel('Scheduled date'); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'coverage.png'); plt.show()


## 4. Signal and sizing

Only use completed closes at decision time t:

$$ r_{i,d,t}=\log\frac{C_{i,d,t-1}}{C_{i,d,t-16}},\qquad x_{i,d,t}=r_{i,d,t}-r_{m,d,t},\qquad z_{i,d,t}=\frac{x_{i,d,t}-\bar{x}_{i,t}^{(20)}}{s_{x,i,t}^{(20)}}. $$

All 16 minute labels are needed for both instruments. In the preceding 20 scheduled sessions, the mean and sample SD will have the same clock time – no older completed days will replace the missing history. A stock may not do as well as the index, even though it may be increasing in value. The z score is not a tail probability.

The barrier scale is the stock-only SD $v=s_{r,i,t}^{(20)}$. At entry open P, stop = $Pe^{-v}$ and target = $Pe^{1.5v}$. All policies have the delayed entry and expiry: a decision taken at 09:45 expires at 10:01 after having been entered at 09:46. Notional is INR 10,000. Before the time of expiration, early exits do not generate opportunities or release capital.


### Signal, execution and accounting

The next cells implement the complete method without hidden project scripts. Primary comparisons reset capital each session and use common complete-case dates. The separate funded account carries cash and reserves early proceeds until expiry.

If the inputs are known to be missing before entry, then there is a logged funded abstention. The identified record is terminated by unknown post entry prices and a funding failure is terminal. The base for each evaluation partition is at C₀. Restarting demands a new segment marked as such and is not allowed after economic failure. No equity is ever bridged if it is unknown.


### Configuration and session data


In [ ]:
from collections import Counter, deque
from contextlib import closing
from datetime import date, datetime, time, timedelta, timezone
from math import exp, fsum, isfinite, log
from pathlib import Path
from statistics import mean, stdev
import sqlite3

IST = timezone(timedelta(hours=5, minutes=30))
MINUTE = timedelta(minutes=1)
POLICIES = ("time_only", "stop_only", "stop_and_target")
PATHS = ("O-L-H-C", "O-H-L-C")
SCENARIOS = tuple((policy, path) for path in PATHS for policy in POLICIES)


def calendar_date(value):
    if isinstance(value, datetime):
        return value.date()
    return value if isinstance(value, date) else date.fromisoformat(str(value)[:10])


def research_parameters(cfg):
    """A single declared scenario; this function performs no parameter selection."""
    p = {
        "universe": tuple(cfg["universe"]), "index": cfg.get("index_symbol", ".NSEI"),
        "C0": float(cfg.get("reference_capital_inr", 100000)),
        "allocation": float(cfg.get("allocation_fraction", 0.5)),
        "lookback": int(cfg.get("prior_sessions", 20)),
        "window": int(cfg.get("signal_window_minutes", 15)),
        "threshold": float(cfg.get("z_threshold", -2)),
        "first": int(cfg.get("decision_minutes_after_open", 30)),
        "interval": int(cfg.get("decision_interval_minutes", 30)),
        "holding": int(cfg.get("holding_minutes", 15)),
        "delay": int(cfg.get("delay_minutes", 1)),
        "cost": float(cfg.get("cost_bp", 15)),
        "slippage": float(cfg.get("stop_slippage_bp", 0)),
        "label": cfg.get("timestamp_label", "start"),
        "open": time.fromisoformat(cfg.get("normal_session_open", "09:15")),
        "close": time.fromisoformat(cfg.get("normal_session_close", "15:30")),
    }
    if (not p["universe"] or len(set(p["universe"])) != len(p["universe"])
            or p["index"] in p["universe"]):
        raise ValueError("Declare a distinct stock universe and separate index")
    if p["label"] not in ("start", "end"):
        raise ValueError("timestamp_label must be start or end; neither confirms supplier semantics")
    if not all(isfinite(p[k]) for k in ("C0", "allocation", "threshold", "cost", "slippage")):
        raise ValueError("Parameters must be finite")
    if p["C0"] <= 0 or not 0 < p["allocation"] < 1 or min(p["cost"], p["slippage"]) < 0:
        raise ValueError("Invalid capital, allocation or costs")
    if (p["lookback"] < 2 or min(p["window"], p["delay"], p["holding"]) < 1
            or p["first"] < p["window"] + 1 or p["interval"] < p["holding"]):
        raise ValueError("Invalid history, timing or overlapping scheduled holding windows")
    if p["open"] >= p["close"]:
        raise ValueError("Invalid session bounds")
    p["notional"] = p["allocation"] * p["C0"] / len(p["universe"])
    return p


def session_bounds(day, p):
    return (datetime.combine(day, p["open"], IST),
            datetime.combine(day, p["close"], IST))


def read_session(connection, day, p):
    """Read only one day/symbol at a time. SQLite stores unchanged naive source labels."""
    start, end = session_bounds(day, p)
    output = {}
    for symbol in (*p["universe"], p["index"]):
        values = {}
        rows = connection.execute(
            "SELECT stamp,o,h,l,c FROM bars WHERE symbol=? AND stamp>=? AND stamp<? ORDER BY stamp",
            (symbol, day.isoformat() + "T00:00:00", (day + timedelta(days=1)).isoformat() + "T00:00:00"))
        for raw, opening, high, low, close in rows:
            stamp = datetime.fromisoformat(raw)
            if stamp.tzinfo is not None or stamp.second or stamp.microsecond:
                raise ValueError("Expected naive, exact-minute source labels")
            stamp = stamp.replace(tzinfo=IST) - (MINUTE if p["label"] == "end" else timedelta(0))
            prices = tuple(float(x) for x in (opening, high, low, close))
            o, h, l, c = prices
            if not all(isfinite(x) and x > 0 for x in prices) or not l <= min(o, c) <= max(o, c) <= h:
                raise ValueError(f"Invalid OHLC: {symbol} {raw}")
            if stamp in values:
                raise ValueError(f"Duplicate transformed label: {symbol} {stamp}")
            if start <= stamp < end:
                values[stamp] = {"o": o, "h": h, "l": l, "c": c}
        output[symbol] = values
    return output

### Past-only signal and entry schedule


In [ ]:
def scheduled_decisions(day, p):
    opening, closing = session_bounds(day, p)
    at = opening + p["first"] * MINUTE
    decisions = []
    while at + (p["delay"] + p["holding"]) * MINUTE < closing:
        decisions.append(at)
        at += p["interval"] * MINUTE
    return decisions


def window_feature(stock, index, decision, window):
    labels = [decision - k * MINUTE for k in range(window + 1, 0, -1)]
    if any(at not in stock or at not in index for at in labels):
        return None
    rs = log(stock[labels[-1]]["c"] / stock[labels[0]]["c"])
    ri = log(index[labels[-1]]["c"] / index[labels[0]]["c"])
    return (rs, rs - ri)


def plan_session(day, bars, history, p):
    """History contains exactly preceding scheduled sessions, including missing maps."""
    features, candidates, episodes = {}, [], []
    index = bars.get(p["index"], {})
    for decision in scheduled_decisions(day, p):
        for symbol in p["universe"]:
            key = (symbol, decision.time())
            value = window_feature(bars.get(symbol, {}), index, decision, p["window"])
            features[key] = value
            row = {"date": day.isoformat(), "symbol": symbol, "decision_at": decision.isoformat(),
                   "status": "MISSING_CURRENT_WINDOW", "z": None, "stock_volatility": None}
            if value is not None:
                row.update(stock_return=value[0], relative_return=value[1])
                if len(history) < p["lookback"]:
                    row["status"] = "WARMUP"
                else:
                    prior = [entry[1].get(key) for entry in history]
                    if any(x is None for x in prior):
                        row["status"] = "MISSING_HISTORY_WINDOW"
                    else:
                        relative_mean = mean(x[1] for x in prior)
                        relative_sd = stdev(x[1] for x in prior)
                        stock_sd = stdev(x[0] for x in prior)
                        row.update(prior_relative_mean=relative_mean, prior_relative_sd=relative_sd,
                                   stock_volatility=stock_sd)
                        if relative_sd == 0 or stock_sd == 0:
                            row["status"] = "UNDEFINED_SCALE"
                        else:
                            row["z"] = (value[1] - relative_mean) / relative_sd
                            row["status"] = "SIGNAL" if row["z"] <= p["threshold"] else "NO_SIGNAL"
            candidates.append(row)
            if row["status"] == "SIGNAL":
                entry = decision + p["delay"] * MINUTE
                episodes.append({"episode_id": f"{day}_{symbol}_{decision:%H%M}", "symbol": symbol,
                                 "signal_at": decision, "entry_at": entry,
                                 "expiry_at": entry + p["holding"] * MINUTE,
                                 "stop_multiplier": exp(-row["stock_volatility"]),
                                 "target_multiplier": exp(1.5 * row["stock_volatility"]),
                                 "notional": p["notional"]})
    return features, candidates, episodes

### Execution and account balances


In [ ]:
def fresh_accounts(capital):
    return {key: {"cash": float(capital), "positions": {}, "reserved": {}, "peak": float(capital),
                  "last_equity": None, "last_cash": None, "last_reserved": None}
            for key in SCENARIOS}


def barrier_fill(position, bar, at, policy, path):
    """At expiry only its opening price matters. No later high/low can precede that exit."""
    if at == position["expiry_at"]:
        return "TIME", bar["o"], "open", False
    if policy == "time_only":
        return None
    stop, target = position["stop"], position["target"]
    if bar["o"] <= stop:
        return "STOP", bar["o"], "open", False
    if policy == "stop_and_target" and bar["o"] >= target:
        return "TARGET", target, "open", False
    stop_hit = bar["l"] <= stop
    target_hit = policy == "stop_and_target" and bar["h"] >= target
    if stop_hit and target_hit:
        return ("STOP", stop, "within_bar", True) if path == "O-L-H-C" else ("TARGET", target, "within_bar", True)
    if stop_hit:
        return "STOP", stop, "within_bar", False
    if target_hit:
        return "TARGET", target, "within_bar", False
    return None


def release_cash(account, at):
    released = []
    for episode_id, (expiry, amount) in list(account["reserved"].items()):
        if expiry <= at:
            del account["reserved"][episode_id]
            released.append((episode_id, amount))
    return released


def available_cash(account):
    return account["cash"] - fsum(amount for _, amount in account["reserved"].values())


def close_position(account, position, at, fill, p, policy, path, accounting, segment):
    reason, raw_price, phase, ambiguous = fill
    extra = p["slippage"] if reason == "STOP" else 0.0
    exit_price = raw_price - position["entry_price"] * extra / 10000
    if exit_price <= 0:
        raise ValueError("Declared adverse slippage produces a nonpositive exit price")
    exit_fee = position["notional"] * p["cost"] / 20000
    proceeds = position["shares"] * exit_price - exit_fee
    account["cash"] += proceeds
    if at < position["expiry_at"] and proceeds > 0:
        account["reserved"][position["episode_id"]] = (position["expiry_at"], proceeds)
    del account["positions"][position["symbol"]]
    net_pnl = proceeds - position["notional"] - position["entry_fee"]
    held = (at - position["entry_at"]).total_seconds() / 60
    return {"date": at.date().isoformat(), "accounting": accounting, "segment": segment,
            "policy": policy, "path": path, "episode_id": position["episode_id"],
            "symbol": position["symbol"], "status": "RESOLVED",
            "signal_at": position["signal_at"].isoformat(), "entry_at": position["entry_at"].isoformat(),
            "expiry_at": position["expiry_at"].isoformat(), "exit_at": at.isoformat(), "exit_phase": phase,
            "entry_price": position["entry_price"], "stop_price": position["stop"], "target_price": position["target"],
            "exit_price": exit_price, "exit_reason": reason, "notional": position["notional"],
            "entry_fee": position["entry_fee"], "exit_fee": exit_fee,
            "extra_stop_slippage_bp": extra, "net_pnl": net_pnl,
            "net_return_bp": net_pnl / position["notional"] * 10000,
            "ambiguous": ambiguous, "holding_lower_minutes": held,
            "holding_upper_minutes": held + (1 if phase == "within_bar" else 0)}


def replay_session(day, bars, episodes, accounts, p, accounting, segment, equity_sink=None):
    """Six matched accounts; no hidden fills, borrowing, capital recycling or automatic resets.

    An unknown required bar ends the common identified record BEFORE that minute's events.
    Flat periods need no market price. Every minute close is valued; identical consecutive
    valuations are compressed. Therefore drawdown is unchanged, but this is not tick risk.
    """
    opening, closing = session_bounds(day, p)
    start_cash = {key: a["cash"] for key, a in accounts.items()}
    if any(a["positions"] or a["reserved"] for a in accounts.values()):
        raise ValueError("A new session requires flat, identified accounts with released cash")
    by_entry = {}
    for ep in episodes:
        if (not opening <= ep["entry_at"] < ep["expiry_at"] < closing
                or ep["entry_at"] - ep["signal_at"] < MINUTE):
            raise ValueError("Episode timing is outside the declared intraday schedule")
        by_entry.setdefault(ep["entry_at"], []).append(ep)
    previous_expiry = None
    for at, batch in sorted(by_entry.items()):
        if (len({ep["symbol"] for ep in batch}) != len(batch)
                or len({ep["expiry_at"] for ep in batch}) != 1
                or (previous_expiry is not None and at < previous_expiry)):
            raise ValueError("Common episode batches must be nonoverlapping and have one entry per stock")
        previous_expiry = batch[0]["expiry_at"]
    trades, equity, cash_events, failure = [], [], [], None
    minimum_cash = {key: available_cash(a) for key, a in accounts.items()}

    def execute_exit(key, account, position, at, fill):
        trade = close_position(account, position, at, fill, p, *key, accounting, segment)
        trades.append(trade)
        cash_events.append({"date": day.isoformat(), "timestamp": at.isoformat(),
                            "accounting": accounting, "segment": segment, "policy": key[0], "path": key[1],
                            "event": "EXIT_" + trade["exit_reason"], "episode_id": trade["episode_id"],
                            "phase": trade["exit_phase"],
                            "cash_delta": position["shares"] * trade["exit_price"] - trade["exit_fee"],
                            "available_cash": available_cash(account)})

    def emit_equity(at, kind, force=False):
        for (policy, path), account in accounts.items():
            market_value = fsum(pos["shares"] * bars[symbol][at - MINUTE]["c"]
                                for symbol, pos in account["positions"].items()) if kind == "minute_close" else 0.0
            value = account["cash"] + market_value
            reserved = fsum(x[1] for x in account["reserved"].values())
            account["peak"] = max(account["peak"], value)
            changed = (value, account["cash"], reserved) != (account["last_equity"], account["last_cash"], account["last_reserved"])
            if force or changed:
                row = {"date": day.isoformat(), "timestamp": at.isoformat(), "accounting": accounting,
                       "segment": segment, "policy": policy, "path": path, "sampling": kind,
                       "cash": account["cash"], "available_cash": available_cash(account),
                       "reserved_cash": reserved, "market_value": market_value, "equity": value,
                       "drawdown_fraction": 1 - value / account["peak"],
                       "open_positions": len(account["positions"])}
                if equity_sink is None:
                    equity.append(row)
                else:
                    equity_sink(row)
            account["last_equity"], account["last_cash"], account["last_reserved"] = value, account["cash"], reserved

    if accounting == "funded":
        emit_equity(opening, "session_open", True)
    at = opening
    while at < closing:
        new = by_entry.get(at, [])
        required = {ep["symbol"] for ep in new}
        required.update(symbol for a in accounts.values() for symbol in a["positions"])
        absent = sorted(symbol for symbol in required if at not in bars.get(symbol, {}))
        if absent:
            failure = {"status": "UNKNOWN_EQUITY", "at": at.isoformat(), "symbols": absent,
                       "reason": "Missing required entry, exit or held-position valuation bar"}
            for key in accounts:
                for ep in new:
                    trades.append({"date": day.isoformat(), "accounting": accounting, "segment": segment,
                                   "policy": key[0], "path": key[1], "episode_id": ep["episode_id"],
                                   "symbol": ep["symbol"], "status": "UNRESOLVED_ENTRY", "net_pnl": None,
                                   "unresolved_at": at.isoformat(), "reason": failure["reason"]})
            break
        for key, account in accounts.items():
            released = release_cash(account, at)
            if released:
                cash_events.append({"date": day.isoformat(), "timestamp": at.isoformat(),
                                    "accounting": accounting, "segment": segment, "policy": key[0], "path": key[1],
                                    "event": "COMMON_EXPIRY_RELEASE", "cash_delta": 0.0,
                                    "released_cash": fsum(amount for _, amount in released),
                                    "available_cash": available_cash(account)})
            # Earlier expiry exits precede any new entries and all later intraminute extremes.
            for position in list(account["positions"].values()):
                if at == position["expiry_at"]:
                    execute_exit(key, account, position, at,
                                 ("TIME", bars[position["symbol"]][at]["o"], "open", False))
        if new:
            required_cash = fsum(ep["notional"] * (1 + p["cost"] / 20000) for ep in new)
            shortages = [{"policy": key[0], "path": key[1], "available_cash": available_cash(a),
                          "required_cash": required_cash, "known_equity": a["cash"]}
                         for key, a in accounts.items() if available_cash(a) + 1e-8 < required_cash]
            if shortages:
                failure = {"status": "FUNDING_FAILURE", "at": at.isoformat(), "accounts": shortages,
                           "reason": "A matched account cannot fund all common entries including fees"}
                if accounting == "funded":
                    # Adjacent batches may have closed prior positions at this open.
                    # Record that known final cash instead of leaving a stale last mark.
                    if any(a["positions"] for a in accounts.values()):
                        raise AssertionError("Funding failure boundary must be flat after common expiry")
                    emit_equity(at, "funding_failure_open", True)
                break
            for key, account in accounts.items():
                for ep in new:
                    if ep["symbol"] in account["positions"]:
                        raise ValueError("Overlapping same-stock entry or forbidden early re-entry")
                    price = bars[ep["symbol"]][at]["o"]
                    fee = ep["notional"] * p["cost"] / 20000
                    account["cash"] -= ep["notional"] + fee
                    account["positions"][ep["symbol"]] = dict(ep, entry_price=price,
                        shares=ep["notional"] / price, entry_fee=fee,
                        stop=price * ep["stop_multiplier"], target=price * ep["target_multiplier"])
                cash_events.append({"date": day.isoformat(), "timestamp": at.isoformat(),
                                    "accounting": accounting, "segment": segment, "policy": key[0], "path": key[1],
                                    "event": "COMMON_ENTRY", "required_cash": required_cash,
                                    "cash_delta": -required_cash,
                                    "available_cash": available_cash(account)})
        exit_cost_shortages = []
        for key, account in accounts.items():
            minimum_cash[key] = min(minimum_cash[key], available_cash(account))
            for position in list(account["positions"].values()):
                fill = barrier_fill(position, bars[position["symbol"]][at], at, *key)
                if fill is not None:
                    execute_exit(key, account, position, at, fill)
            minimum_cash[key] = min(minimum_cash[key], available_cash(account))
            if available_cash(account) < -1e-7 or account["cash"] < -1e-7:
                known_value = account["cash"] + fsum(pos["shares"] * bars[symbol][at]["c"]
                                                     for symbol, pos in account["positions"].items())
                exit_cost_shortages.append({"policy": key[0], "path": key[1],
                                            "available_cash": available_cash(account), "cash": account["cash"],
                                            "additional_funding_required": max(0.0, -available_cash(account)),
                                            "known_equity_at_bar_close": known_value})
        if exit_cost_shortages:
            failure = {"status": "FUNDING_FAILURE", "at": at.isoformat(),
                       "accounts": exit_cost_shortages, "phase": "bar_outcome",
                       "reason": "Realised exit costs create a cash deficit; terminal economic failure, no continued borrowing"}
            if accounting == "funded":
                emit_equity(at + MINUTE, "minute_close", True)
            break
        if accounting == "funded":
            emit_equity(at + MINUTE, "minute_close", force=(at + MINUTE == closing))
        at += MINUTE
    if failure is not None:
        for key, account in accounts.items():
            for pos in account["positions"].values():
                trades.append({"date": day.isoformat(), "accounting": accounting, "segment": segment,
                               "policy": key[0], "path": key[1], "episode_id": pos["episode_id"],
                               "symbol": pos["symbol"],
                               "status": "UNRESOLVED" if failure["status"] == "UNKNOWN_EQUITY" else "OPEN_AT_FUNDING_FAILURE",
                               "net_pnl": None,
                               "unresolved_at": failure["at"], "reason": failure["reason"]})
    else:
        for account in accounts.values():
            release_cash(account, closing)
        if any(a["positions"] or a["reserved"] for a in accounts.values()):
            raise AssertionError("Normal session ended with an open position or reserved proceeds")
    daily = []
    for key, account in accounts.items():
        pnl = None if failure else account["cash"] - start_cash[key]
        ledger_pnl = fsum(r["net_pnl"] for r in trades if (r["policy"], r["path"]) == key and r["status"] == "RESOLVED")
        cash_change = fsum(r["cash_delta"] for r in cash_events if (r["policy"], r["path"]) == key)
        if abs(account["cash"] - start_cash[key] - cash_change) > max(1e-7, abs(account["cash"]) * 1e-12):
            raise AssertionError("Cash flow ledger does not reconcile to physical cash")
        if failure is None and abs(pnl - ledger_pnl) > max(1e-7, abs(account["cash"]) * 1e-12):
            raise AssertionError("Cash change does not reconcile to realised trade P&L")
        daily.append({"date": day.isoformat(), "accounting": accounting, "segment": segment,
                      "policy": key[0], "path": key[1], "status": failure["status"] if failure else "EVALUABLE",
                      "net_pnl": pnl, "return_on_reference_capital": None if pnl is None else pnl / p["C0"],
                      "opening_cash": start_cash[key], "closing_cash": None if failure else account["cash"],
                      "minimum_available_cash": minimum_cash[key],
                      "unique_planned_episodes": len(episodes), "failure": failure})
    return {"daily": daily, "trades": trades, "equity": equity, "cash_events": cash_events, "failure": failure}

### Session replay and funded segments


In [ ]:
def run_research(DB_PATH, CFG, CALENDAR, evaluation_dates, *, equity_sink=None):
    """No market outcomes are read outside the supplied calendar slice.

    evaluation_dates must be a predeclared contiguous calendar slice, not selected complete days.
    Use a fresh invocation for validation and final test: each funded diagnostic starts at C0.
    The only allowed later new-C0 restart is explicit CFG['funded_restart_dates']={date: reason}
    after unknown data. It is a NEW segment; an economic funding failure is terminal.
    """
    p = research_parameters(CFG)
    calendar = [calendar_date(x) for x in CALENDAR]
    if calendar != sorted(set(calendar)):
        raise ValueError("CALENDAR must be unique and strictly increasing")
    evaluate = {calendar_date(x) for x in evaluation_dates}
    if not evaluate or not evaluate <= set(calendar):
        raise ValueError("evaluation_dates must be a nonempty subset of the scheduled calendar")
    selected = [day for day in calendar if min(evaluate) <= day <= max(evaluate)]
    if set(selected) != evaluate:
        raise ValueError("Use a contiguous calendar slice; do not select dates by future completeness")
    calendar = [day for day in calendar if day <= max(evaluate)]
    excluded = {calendar_date(x) for x in CFG.get("excluded_nonstandard_sessions", ())}
    restarts = {calendar_date(k): v for k, v in CFG.get("funded_restart_dates", {}).items()}
    if not set(restarts) <= evaluate or any(not str(v).strip() for v in restarts.values()):
        raise ValueError("Every explicit restart needs an evaluation date and reason")
    history = deque(maxlen=p["lookback"])
    output = {name: [] for name in ("daily", "trades", "equity", "cash_events", "diagnostics", "candidates", "segments")}
    funded, funded_status, segment = None, "NOT_STARTED", 0
    uri = Path(DB_PATH).resolve().as_uri() + "?mode=ro"
    with closing(sqlite3.connect(uri, uri=True)) as connection:
        connection.execute("PRAGMA cache_size=-2048")
        connection.execute("PRAGMA temp_store=FILE")
        for day in calendar:
            bars = {} if day in excluded else read_session(connection, day, p)
            features, candidates, episodes = plan_session(day, bars, history, p)
            if day in evaluate:
                if funded is None:
                    segment += 1
                    funded = fresh_accounts(p["C0"])
                    funded_status = "ACTIVE"
                    output["segments"].append({"segment": segment, "start_date": day.isoformat(),
                                               "initial_capital": p["C0"], "reason": "Predeclared evaluation boundary"})
                elif day in restarts:
                    if funded_status != "UNKNOWN_EQUITY":
                        raise ValueError("A restart is allowed only after unknown data, never a funding failure or active segment")
                    segment += 1
                    funded = fresh_accounts(p["C0"])
                    funded_status = "ACTIVE"
                    output["segments"].append({"segment": segment, "start_date": day.isoformat(),
                                               "initial_capital": p["C0"], "reason": str(restarts[day])})
                counts = Counter(row["status"] for row in candidates)
                invalid = [row for row in candidates if row["status"] not in ("SIGNAL", "NO_SIGNAL")]
                primary_status = ("EXCLUDED_SESSION" if day in excluded else
                                  "INCOMPLETE_CANDIDATE_COVERAGE" if invalid or not candidates else "EVALUABLE")
                if day in excluded:
                    episodes = []
                independent = replay_session(day, bars, episodes, fresh_accounts(p["C0"]), p,
                                             "independent", 0)
                if independent["failure"] is not None:
                    primary_status = independent["failure"]["status"]
                if primary_status != "EVALUABLE":
                    for row in independent["daily"]:
                        row.update(status=primary_status, net_pnl=None, return_on_reference_capital=None, closing_cash=None)
                for row in independent["trades"]:
                    row["included_primary_day"] = primary_status == "EVALUABLE"
                for name in ("daily", "trades", "cash_events"):
                    output[name].extend(independent[name])
                if funded_status == "ACTIVE":
                    # Known unavailable pre-entry inputs create abstentions. No unknown prices are imputed.
                    current = replay_session(day, bars, episodes, funded, p, "funded", segment, equity_sink)
                    for name in ("daily", "trades", "equity", "cash_events"):
                        output[name].extend(current[name])
                    if current["failure"] is not None:
                        funded_status = current["failure"]["status"]
                        output["segments"][-1].update(end_date=day.isoformat(), end_reason=funded_status,
                                                      failure=current["failure"])
                else:
                    for policy, path in SCENARIOS:
                        output["daily"].append({"date": day.isoformat(), "accounting": "funded", "segment": segment,
                                                "policy": policy, "path": path, "status": "BLOCKED_" + funded_status,
                                                "net_pnl": None, "return_on_reference_capital": None,
                                                "opening_cash": None, "closing_cash": None})
                output["diagnostics"].append({"date": day.isoformat(), "candidate_counts": dict(counts),
                                              "known_abstentions": len(invalid), "planned_episodes": len(episodes),
                                              "primary_status": primary_status, "funded_status": funded_status,
                                              "segment": segment, "deliberately_excluded": day in excluded})
                if CFG.get("store_candidates", False):
                    output["candidates"].extend(candidates)
                else:
                    output["candidates"].extend(row for row in candidates if row["status"] == "SIGNAL")
            # Appended AFTER today's signals: no current or future observations enter their own history.
            history.append((day, features))
    if output["segments"] and "end_date" not in output["segments"][-1]:
        output["segments"][-1].update(end_date=max(evaluate).isoformat(), end_reason="END_OF_EVALUATION")
    output["metadata"] = {"title": "Path-Robust Risk Controls for Intraday Mean Reversion in NSE Equities",
                          "timestamp_label_assumption": p["label"], "timezone_assumption": "Asia/Kolkata",
                          "C0": p["C0"], "cost_bp": p["cost"], "stop_slippage_bp": p["slippage"],
                          "delay_minutes": p["delay"], "evaluation_start": min(evaluate).isoformat(),
                          "evaluation_end": max(evaluate).isoformat(), "terminal_funded_status": funded_status,
                          "equity_sampling": "Minute closes plus known flat opening/failure boundaries; unchanged consecutive values omitted",
                          "primary_estimand": "Complete-case independent-session P&L difference / C0",
                          "funded_estimand": "Separate self-financing fixed-notional feasibility diagnostic",
                          "notes": ["No target-only policy; no parameter search; no automatic capital reset.",
                                    "Explicit excluded sessions are retrospective sample restrictions, not outage-loss estimates.",
                                    "Intraminute fills are assumed and their exact execution seconds are unknown."]}
    return output

### Synthetic method checks


In [ ]:
def run_notebook_method_checks():
    """Synthetic checks exercise semantics. They supply no market evidence or holdout result."""
    p = research_parameters({"universe": ["STOCK"], "reference_capital_inr": 1000, "cost_bp": 0})
    day = date(2021, 1, 4)
    start = datetime.combine(day, time(9, 46), IST)
    ep = {"episode_id": "synthetic", "symbol": "STOCK", "signal_at": start - MINUTE,
          "entry_at": start, "expiry_at": start + 15 * MINUTE,
          "stop_multiplier": 0.99, "target_multiplier": 1.015, "notional": 500.0}
    flat = {start + i * MINUTE: {"o": 100.0, "h": 100.0, "l": 100.0, "c": 100.0} for i in range(16)}
    both = dict(flat)
    both[start] = {"o": 100.0, "h": 102.0, "l": 98.0, "c": 100.0}
    accounts = fresh_accounts(1000)
    result = replay_session(day, {"STOCK": both}, [ep], accounts, p, "funded", 1)
    found = {(r["policy"], r["path"]): r for r in result["trades"]}
    assert found[("stop_and_target", "O-L-H-C")]["exit_reason"] == "STOP"
    assert found[("stop_and_target", "O-H-L-C")]["exit_reason"] == "TARGET"
    assert found[("time_only", "O-L-H-C")]["exit_reason"] == "TIME"
    assert found[("stop_and_target", "O-L-H-C")]["ambiguous"]
    before_expiry = [r for r in result["equity"] if r["policy"] == "stop_and_target" and r["path"] == "O-H-L-C"
                     and r["timestamp"] == (start + MINUTE).isoformat()][0]
    assert abs(before_expiry["available_cash"] - 500) < 1e-9
    assert abs(before_expiry["reserved_cash"] - 507.5) < 1e-9
    assert abs(before_expiry["equity"] - 1007.5) < 1e-9
    assert not accounts[("stop_and_target", "O-H-L-C")]["reserved"]
    # The time-only position marks its changing close; no cash-only fake drawdown.
    marked = dict(flat)
    marked[start] = {"o": 100.0, "h": 100.0, "l": 98.0, "c": 98.0}
    marked_result = replay_session(day, {"STOCK": marked}, [ep], fresh_accounts(1000), p, "funded", 1)
    first_mark = next(r for r in marked_result["equity"] if r["policy"] == "time_only" and r["path"] == "O-L-H-C"
                      and r["timestamp"] == (start + MINUTE).isoformat())
    assert abs(first_mark["equity"] - 990) < 1e-9 and abs(first_mark["drawdown_fraction"] - 0.01) < 1e-9
    # Expiry open wins even if later extrema touch both barriers.
    expiry = dict(flat)
    expiry[ep["expiry_at"]] = {"o": 100.25, "h": 110.0, "l": 90.0, "c": 100.0}
    expired = replay_session(day, {"STOCK": expiry}, [ep], fresh_accounts(1000), p, "independent", 0)
    assert all(r["exit_reason"] == "TIME" and r["exit_price"] == 100.25 for r in expired["trades"])
    # Opening gap at a stop, with slippage debited once.
    gap = dict(flat)
    gap[start + MINUTE] = {"o": 98.0, "h": 98.0, "l": 98.0, "c": 98.0}
    slip_p = dict(p, slippage=5.0)
    gapped = replay_session(day, {"STOCK": gap}, [ep], fresh_accounts(1000), slip_p, "independent", 0)
    stopped = next(r for r in gapped["trades"] if r["policy"] == "stop_only")
    assert abs(stopped["exit_price"] - 97.95) < 1e-9 and abs(stopped["net_pnl"] + 10.25) < 1e-9
    # Missing held-position bar halts common equity; it is not a zero or interpolated price.
    missing = dict(flat)
    del missing[start + MINUTE]
    unresolved = replay_session(day, {"STOCK": missing}, [ep], fresh_accounts(1000), p, "funded", 1)
    assert unresolved["failure"]["status"] == "UNKNOWN_EQUITY"
    assert all(r["net_pnl"] is None for r in unresolved["daily"])
    # A funding shortage in ONE account prevents entries in ALL matched accounts.
    short = fresh_accounts(1000)
    short[("stop_only", "O-L-H-C")]["cash"] = 499.0
    failed = replay_session(day, {"STOCK": flat}, [ep], short, p, "funded", 1)
    assert failed["failure"]["status"] == "FUNDING_FAILURE" and not failed["trades"]
    assert all(not a["positions"] for a in short.values())
    # An adjacent expiry-open gain must be marked before a new batch fails funding.
    adjacent_bars = dict(flat)
    adjacent_bars[ep["expiry_at"]] = {"o": 100.25, "h": 100.25, "l": 100.25, "c": 100.25}
    adjacent_ep = dict(ep, episode_id="adjacent", signal_at=ep["expiry_at"] - MINUTE,
                       entry_at=ep["expiry_at"], expiry_at=ep["expiry_at"] + 15 * MINUTE, notional=1002.0)
    adjacent = replay_session(day, {"STOCK": adjacent_bars}, [ep, adjacent_ep], fresh_accounts(1000), p, "funded", 1)
    final_mark = next(r for r in reversed(adjacent["equity"]) if r["policy"] == "time_only" and r["path"] == "O-L-H-C")
    assert adjacent["failure"]["status"] == "FUNDING_FAILURE"
    assert final_mark["sampling"] == "funding_failure_open" and abs(final_mark["equity"] - 1001.25) < 1e-9
    # A realised exit-fee deficit is an economic failure, not an assertion or hidden loan.
    crashed_bars = dict(flat)
    crashed_bars[start + MINUTE] = {"o": 0.001, "h": 0.001, "l": 0.001, "c": 0.001}
    thin_cash = fresh_accounts(1000)
    for account in thin_cash.values():
        account["cash"] = 500.375  # Exactly covers INR500 entry and half of 15 bp cost.
    crashed = replay_session(day, {"STOCK": crashed_bars}, [ep], thin_cash, dict(p, cost=15.0), "funded", 1)
    assert crashed["failure"]["status"] == "FUNDING_FAILURE"
    assert "cash deficit" in crashed["failure"]["reason"]
    cash_deficit = next(a for a in crashed["failure"]["accounts"] if a["policy"] == "stop_only")
    assert abs(cash_deficit["cash"] + 0.37) < 1e-9
    assert abs(cash_deficit["additional_funding_required"] - 0.37) < 1e-9
    assert any(r["status"] == "RESOLVED" and r["exit_reason"] == "STOP" for r in crashed["trades"])
    assert any(r["status"] == "OPEN_AT_FUNDING_FAILURE" for r in crashed["trades"])
    # A second day carries cash, rather than resetting prior profits to C0.
    next_day = day + timedelta(days=1)
    next_bars = {t + timedelta(days=1): b for t, b in flat.items()}
    next_ep = dict(ep, episode_id="next", signal_at=ep["signal_at"] + timedelta(days=1),
                   entry_at=ep["entry_at"] + timedelta(days=1), expiry_at=ep["expiry_at"] + timedelta(days=1))
    carried = replay_session(next_day, {"STOCK": next_bars}, [next_ep], accounts, p, "funded", 1)
    carry_row = next(r for r in carried["daily"] if r["policy"] == "stop_and_target" and r["path"] == "O-H-L-C")
    assert abs(carry_row["opening_cash"] - 1007.5) < 1e-9
    # A close-to-close feature uses only the sixteen completed labels.
    decision = start - MINUTE
    labels = [decision - k * MINUTE for k in range(16, 0, -1)]
    stock = {t: {"c": 100 + i / 10} for i, t in enumerate(labels)}
    index = {t: {"c": 100} for t in labels}
    original = window_feature(stock, index, decision, 15)
    stock[decision + 10 * MINUTE] = {"c": 1e12}
    assert window_feature(stock, index, decision, 15) == original
    del stock[labels[5]]
    assert window_feature(stock, index, decision, 15) is None
    # A missing prior session stays missing; an earlier valid date cannot replace it.
    hp = dict(p, lookback=2)
    features = {(symbol, t.time()): (0.001, 0.001)
                for t in scheduled_decisions(day, hp) for symbol in hp["universe"]}
    historical = deque([(day - timedelta(days=2), features), (day - timedelta(days=1), {})], maxlen=2)
    current = {"STOCK": {t: {"c": 100} for t in labels}, hp["index"]: index}
    _, candidate_rows, _ = plan_session(day, current, historical, hp)
    assert candidate_rows[0]["status"] == "MISSING_HISTORY_WINDOW"
    # End-to-end lifecycle on a small synthetic SQLite database, not instructor data.
    import tempfile
    with tempfile.TemporaryDirectory() as temporary:
        database = Path(temporary) / "synthetic.sqlite"
        dates = [date(2021, 1, 4) + timedelta(days=i) for i in range(5)]
        with closing(sqlite3.connect(database)) as connection:
            connection.execute("CREATE TABLE bars(symbol TEXT,stamp TEXT,o REAL,h REAL,l REAL,c REAL,volume REAL,source_id INTEGER,PRIMARY KEY(symbol,stamp)) WITHOUT ROWID")
            for synthetic_day, slope in zip(dates, (0.001, -0.001, -0.01, 0.001, -0.001)):
                opening = datetime.combine(synthetic_day, time(9, 15))
                for i in range(375):
                    stamp = opening + i * MINUTE
                    for symbol, rate in (("STOCK", slope), (".NSEI", 0.0)):
                        o = 100 * exp(rate * i / 15)
                        c = 100 * exp(rate * (i + 1) / 15)
                        connection.execute("INSERT INTO bars VALUES(?,?,?,?,?,?,?,?)",
                                           (symbol, stamp.isoformat(), o, max(o, c), min(o, c), c, 1, 1))
            connection.commit()
        cfg = {"universe": ["STOCK"], "reference_capital_inr": 1000, "prior_sessions": 2,
               "cost_bp": 20, "allocation_fraction": 0.99}
        # Validation may use earlier history, but must never read a later outcome day.
        observed_dates = []
        original_reader = globals()["read_session"]
        def bounded_reader(connection, current_day, parameters):
            observed_dates.append(current_day)
            if current_day > dates[3]:
                raise AssertionError("Read beyond the synthetic validation boundary")
            return original_reader(connection, current_day, parameters)
        globals()["read_session"] = bounded_reader
        try:
            bounded = run_research(database, cfg, dates, dates[2:4])
            assert observed_dates == dates[:4]
            assert {r["date"] for r in bounded["daily"]} == {str(d) for d in dates[2:4]}
            try:
                run_research(database, cfg, dates, [dates[2], dates[4]])
            except ValueError as error:
                assert "contiguous" in str(error)
            else:
                raise AssertionError("Noncontiguous evaluation dates were accepted")
        finally:
            globals()["read_session"] = original_reader
        funded_failure = run_research(database, cfg, dates, dates[2:])
        assert funded_failure["metadata"]["terminal_funded_status"] == "FUNDING_FAILURE"
        assert any(r["status"] == "BLOCKED_FUNDING_FAILURE" for r in funded_failure["daily"])
        assert len(funded_failure["segments"]) == 1
        with closing(sqlite3.connect(database)) as connection:
            connection.execute("DELETE FROM bars WHERE symbol='STOCK' AND stamp=?", (dates[2].isoformat() + "T09:47:00",))
            connection.commit()
        cfg.update(allocation_fraction=0.5, cost_bp=0)
        unknown = run_research(database, cfg, dates, dates[2:])
        assert unknown["metadata"]["terminal_funded_status"] == "UNKNOWN_EQUITY"
        assert len(unknown["segments"]) == 1
        assert any(r["status"] == "BLOCKED_UNKNOWN_EQUITY" for r in unknown["daily"])
        restarted = run_research(database, dict(cfg, funded_restart_dates={dates[4].isoformat(): "Synthetic independent segment"}), dates, dates[2:])
        assert len(restarted["segments"]) == 2
        restart_rows = [r for r in restarted["daily"] if r["accounting"] == "funded" and r["segment"] == 2]
        assert all(r["opening_cash"] == 1000 for r in restart_rows)
        assert restarted["segments"][1]["reason"] == "Synthetic independent segment"
        # Normalisation changes labels only; no actual supplier convention is inferred.
        with closing(sqlite3.connect(database)) as connection:
            raw_start = read_session(connection, dates[0], research_parameters(cfg))
            raw_end = read_session(connection, dates[0], research_parameters(dict(cfg, timestamp_label="end")))
        first = datetime.combine(dates[0], time(9, 15), IST)
        assert raw_end["STOCK"][first] == raw_start["STOCK"][first + MINUTE]
    return {"synthetic_scenarios": 19, "status": "passed", "market_evidence": False}

METHOD_CHECKS = run_notebook_method_checks()
METHOD_CHECKS

### Worked synthetic examples

The first example shows the calculation of the signal. The second fixes entry at 100, stop at 99 and target at 101 while one candle reaches 98 and 102. It is just the order of the prices that change. These are not market observations, but arithmetic checks.


In [ ]:
past_stock = np.array([.001*np.sin(i)+.0002*i for i in range(20)])
past_index = np.array([.0005*np.cos(i) for i in range(20)])
past_relative = past_stock-past_index
current_stock, current_index = -.009, -.001
worked = dict(stock_return=current_stock,index_return=current_index,index_adjusted_return=current_stock-current_index,
              prior_relative_mean=past_relative.mean(),prior_relative_sample_sd=past_relative.std(ddof=1),
              stock_barrier_scale=past_stock.std(ddof=1))
worked['z'] = (worked['index_adjusted_return']-worked['prior_relative_mean'])/worked['prior_relative_sample_sd']
worked['stop_at_entry_100'] = 100*np.exp(-worked['stock_barrier_scale'])
worked['target_at_entry_100'] = 100*np.exp(1.5*worked['stock_barrier_scale'])
display(pd.DataFrame(worked.items(),columns=['synthetic quantity','value']))
assert worked['z'] < -2 and np.isfinite(worked['z'])
fig, ax = plt.subplots(figsize=(7,3))
for values,label in [([100,98,102,100],'O-L-H-C: stop first'),([100,102,98,100],'O-H-L-C: target first')]:
    ax.plot(range(4),values,'o-',label=label)
ax.axhline(99,color='crimson',ls='--',label='Stop 99'); ax.axhline(101,color='seagreen',ls='--',label='Target 101')
ax.set(xticks=range(4),xticklabels=['Open','First extreme','Second extreme','Close'],
       ylabel='Synthetic price units',xlabel='Assumed ordering (not elapsed seconds)',title='Synthetic both-hit candle: identical OHLC, different exit')
ax.legend(fontsize=8,ncol=2); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'synthetic_ambiguity.png'); plt.show()


## 5. Paired uncertainty and tail risk

For $R_d^p=\mathrm{P\&L}_d^p/C_0$, estimate On common complete-case dates D, including identified no-trade days, $\widehat\Delta$ is the sum of $R_d^{combined}-R_d^{time}$, with d running over the set D, divided by $|D|$. If we have repeated policies/scenarios, they are not independent observations.

Within the contiguous eligible calendar segments resample whole paired daily vectors in fixed-length moving blocks Sample blocks uniformly with replacement, ensuring the last block is cut so that each segment's length is maintained. Don't connect dates that you can't cover and don't drop residual segments. If any segment is less than block length or the sum of all the floor(segment length / block length) is less than 10, withhold intervals. This is not a sample-size theorem; rather, this is a reporting threshold.

The gap-respecting approach is an adaptation of the fixed-block resampling (Künsch), but not the random length stationary bootstrap (Politis and Romano). Five days is primary; ten and twenty are sensitivities, never selected for significance. Limited coverage due to segment conditioning and endpoint effects.

ES97.5: average on the worst 2.5% of daily loss probability mass with fractional weight on the boundary observation (Acerbi and Tasche) Calculate the ES of each of the policies before subtracting to get the difference in ES; ES of paired differences is a different estimand. Report tail mass, contributing dates and leave-one-day-out sensitivity (few extreme days limit precision) (Yamai and Yoshiba).


In [ ]:
def expected_shortfall(losses, confidence=.975):
    values = np.sort(np.asarray(losses,dtype=float))[::-1]
    if len(values)==0 or not np.isfinite(values).all() or not 0<confidence<1:
        raise ValueError('ES requires a finite nonempty sample and confidence in (0,1)')
    mass = (1-confidence)*len(values)
    weights = np.clip(mass-np.arange(len(values)),0,1)
    return float(np.dot(weights,values)/mass)

def eligible_segments(dates, calendar):
    positions = {str(d):i for i,d in enumerate(calendar)}
    loc = [positions[str(d)] for d in dates]
    if len(set(loc)) != len(loc) or loc != sorted(loc):
        raise ValueError('Eligible dates must be distinct and chronological')
    edges = [0] + [i for i in range(1,len(loc)) if loc[i] != loc[i-1]+1] + [len(loc)]
    return [np.arange(a,b) for a,b in zip(edges[:-1],edges[1:]) if b>a]

def paired_block_interval(wide, calendar, left, right, block_length, spec):
    delta = wide[left].to_numpy()-wide[right].to_numpy()
    segments = eligible_segments(wide.index, calendar)
    lengths = [len(s) for s in segments]
    result = dict(n_days=len(delta),mean_difference_bp=float(np.mean(delta)) if len(delta) else np.nan,
                  block_length=block_length,segment_lengths=str(lengths),
                  support_blocks=sum(n//block_length for n in lengths),lower_bp=np.nan,upper_bp=np.nan)
    if not lengths or min(lengths)<block_length:
        return {**result,'status':'UNAVAILABLE: a segment is shorter than the block length'}
    if result['support_blocks'] < spec['minimum_block_support']:
        return {**result,'status':'UNAVAILABLE: fewer than ten support blocks'}
    matrix = wide[[left,right]].to_numpy(dtype=float)
    if not np.isfinite(matrix).all():
        raise ValueError('Never zero-fill unknown daily outcomes before inference')
    rng = np.random.default_rng(spec['seed'])
    sampled_means = np.empty(spec['repetitions'])
    for j in range(spec['repetitions']):
        indices = []
        for segment in segments:
            starts = rng.integers(0,len(segment)-block_length+1,size=math.ceil(len(segment)/block_length))
            indices.extend(np.concatenate([segment[s:s+block_length] for s in starts])[:len(segment)])
        paired = matrix[np.asarray(indices,dtype=int)]
        sampled_means[j] = np.mean(paired[:,0]-paired[:,1])
    alpha = 1-spec['confidence']
    low, high = np.quantile(sampled_means,[alpha/2,1-alpha/2])
    return {**result,'lower_bp':float(low),'upper_bp':float(high),'status':'AVAILABLE: conditional moving-block percentile interval'}

def tail_rows(series, confidence=.975):
    losses = (-series).sort_values(ascending=False,kind='stable')
    mass = (1-confidence)*len(losses)
    weights = np.clip(mass-np.arange(len(losses)),0,1)
    table = pd.DataFrame({'date':losses.index,'loss_bp':losses.values,'tail_weight_days':weights})
    return table[table.tail_weight_days>0].reset_index(drop=True)

assert abs(expected_shortfall(np.arange(100),.975)-98.2)<1e-12
assert expected_shortfall([2,1],.975)==2
assert abs(expected_shortfall([-5,-2],.975)+2)<1e-12
synthetic_dates = pd.bdate_range('2000-01-03',periods=100).strftime('%Y-%m-%d').tolist()
synthetic_pairs = pd.DataFrame({'a':np.arange(100)/100+2,'b':np.arange(100)/100},index=synthetic_dates)
check_ci = paired_block_interval(synthetic_pairs,synthetic_dates,'a','b',5,INFERENCE)
assert np.allclose([check_ci['lower_bp'],check_ci['upper_bp']],[2,2])
gap_pairs = synthetic_pairs.drop(index=synthetic_dates[2])
assert paired_block_interval(gap_pairs,synthetic_dates,'a','b',5,INFERENCE)['status'].startswith('UNAVAILABLE')
assert np.isclose(expected_shortfall(-synthetic_pairs.a)-expected_shortfall(-synthetic_pairs.b),-2)
print('Inference checks passed: exact fractional tail, negative losses, paired resampling, short-gap withholding.')


def diagnostic_mean_weights(dates, calendar, block_length, inference):
    """Cache the exact existing resampler's mean weights, with endpoint equivalence checks.

    For sampled indices I, mean(x[I]) = sum_j count(I == j) * x[j] / len(I).
    The index draws, segment order and truncation below match paired_block_interval.
    This only replaces repeated arithmetic; no sampling or interval rule is changed.
    """
    n = len(dates)
    zero = pd.DataFrame({'left':np.zeros(n), 'right':np.zeros(n)}, index=dates)
    gate = paired_block_interval(zero, calendar, 'left', 'right', block_length, inference)
    if not np.isfinite(gate['lower_bp']):
        return None, gate, None
    segments = eligible_segments(dates, calendar)
    rng = np.random.default_rng(inference['seed'])
    weights = np.empty((inference['repetitions'], n))
    for j in range(inference['repetitions']):
        indices = []
        for segment in segments:
            starts = rng.integers(0, len(segment)-block_length+1,
                                 size=math.ceil(len(segment)/block_length))
            indices.extend(np.concatenate([segment[s:s+block_length] for s in starts])[:len(segment)])
        weights[j] = np.bincount(indices, minlength=n) / n
    assert np.allclose(weights.sum(axis=1), 1.0, rtol=0, atol=1e-12)
    alpha = 1-inference['confidence']
    probes = [np.linspace(-1, 1, n), np.sin(np.arange(n)/3),
              np.random.default_rng(719).normal(size=n)]
    max_error = 0.0
    for delta in probes:
        common = np.cos(np.arange(n)/7)
        paired = pd.DataFrame({'left':common+delta, 'right':common}, index=dates)
        exact = paired_block_interval(paired, calendar, 'left', 'right', block_length, inference)
        fast = np.quantile(weights @ (paired.left.to_numpy()-paired.right.to_numpy()),
                           [alpha/2, 1-alpha/2])
        error = np.max(np.abs(fast-[exact['lower_bp'], exact['upper_bp']]))
        max_error = max(max_error, float(error))
        if error > 1e-11:
            raise AssertionError('Cached arithmetic differs from the existing resampler; do not use it')
    return weights, gate, max_error

def bootstrap_reliability_diagnostic(daily, development_calendar, inference, diagnostic):
    """Gaussian paired-difference simulations on fixed actual development eligibility masks."""
    from statistics import NormalDist
    calendar = [str(d) for d in development_calendar]
    if calendar != sorted(set(calendar)) or not calendar:
        raise ValueError('Supply the complete ordered development calendar')
    if diagnostic['block_length'] != 5 or inference['repetitions'] != 2000:
        raise ValueError('This diagnostic is specified for five-day blocks and 2,000 resamples')
    locations = {d:i for i,d in enumerate(calendar)}
    masks, metadata = {}, {}
    for label in ['start', 'end']:
        good = daily[(daily.timestamp_label==label) & (daily.setting=='reference')
            & (daily.accounting=='independent') & (daily.status=='EVALUABLE')
            & (daily.path=='O-L-H-C')]
        sets = [set(good.loc[good.policy==p, 'date'].astype(str))
                for p in ['time_only', 'stop_only', 'stop_and_target']]
        if not all(s==sets[0] for s in sets):
            raise ValueError('The actual primary policy masks must agree')
        dates = sorted(sets[0])
        if not set(dates) <= set(calendar):
            raise ValueError('Eligibility contains a date outside development')
        indices = np.array([locations[d] for d in dates], dtype=int)
        weights, gate, error = diagnostic_mean_weights(dates, calendar,
            diagnostic['block_length'], inference)
        masks[label] = (indices, weights, gate)
        clean_gate = {k:None if isinstance(v, float) and not math.isfinite(v) else v
                      for k,v in gate.items()}
        metadata[label] = dict(eligible_dates=dates, resampler_gate=clean_gate,
                               maximum_equivalence_error_bp=error)
    rows = []
    repetitions = diagnostic['replications']
    mu, sd = diagnostic['true_mean_bp'], diagnostic['stationary_sd_bp']
    alpha = 1-inference['confidence']
    for phi in diagnostic['phis']:
        if not -1 < phi < 1:
            raise ValueError('AR(1) must be stationary')
        rng = np.random.default_rng(np.random.SeedSequence(
            [diagnostic['process_seed'], int(round(1000*phi))]))
        values = np.empty((repetitions, len(calendar)))
        values[:,0] = mu + sd*rng.normal(size=repetitions)
        innovations = sd*math.sqrt(1-phi*phi)
        for t in range(1, len(calendar)):
            values[:,t] = mu + phi*(values[:,t-1]-mu) + innovations*rng.normal(size=repetitions)
        for label, (indices, weights, gate) in masks.items():
            record = dict(timestamp_label=label, phi=phi, scheduled_days=len(calendar),
                eligible_days=len(indices), segment_lengths=gate['segment_lengths'],
                replications=repetitions, bootstrap_repetitions=inference['repetitions'],
                block_length=diagnostic['block_length'], support_blocks=gate['support_blocks'],
                unavailable_count=repetitions if weights is None else 0,
                coverage_pct=np.nan, coverage_mcse_pp=np.nan, rejection_pct=np.nan,
                rejection_mcse_pp=np.nan, mean_width_bp=np.nan, width_mcse_bp=np.nan,
                median_width_bp=np.nan, gaussian_oracle_width_bp=np.nan,
                mean_width_over_oracle=np.nan, status=gate['status'])
            if weights is not None:
                bootstrap_means = values[:,indices] @ weights.T
                low, high = np.quantile(bootstrap_means, [alpha/2, 1-alpha/2], axis=1)
                contains = (low<=mu) & (mu<=high)
                rejects = (low>mu) | (high<mu)
                assert np.all(contains == ~rejects)
                coverage, rejection = contains.mean(), rejects.mean()
                widths = high-low
                lags = np.abs(indices[:,None]-indices[None,:])
                oracle_se = sd*math.sqrt(float(np.power(phi,lags).sum()) / len(indices)**2)
                oracle_width = 2*NormalDist().inv_cdf(1-alpha/2)*oracle_se
                record.update(coverage_pct=100*coverage,
                    coverage_mcse_pp=100*math.sqrt(coverage*(1-coverage)/repetitions),
                    rejection_pct=100*rejection,
                    rejection_mcse_pp=100*math.sqrt(rejection*(1-rejection)/repetitions),
                    mean_width_bp=float(widths.mean()),
                    width_mcse_bp=float(widths.std(ddof=1)/math.sqrt(repetitions)),
                    median_width_bp=float(np.median(widths)),
                    gaussian_oracle_width_bp=oracle_width,
                    mean_width_over_oracle=float(widths.mean()/oracle_width))
            rows.append(record)
    return pd.DataFrame(rows), dict(specification=diagnostic, inference=inference,
        development_calendar=calendar, masks=metadata,
        interpretation='Synthetic repeated-sample coverage conditional on these masks and the fixed bootstrap seed; not estimated coverage of market intervals.')


def reconcile_ledgers(directory, tolerance=1e-7):
    """Independently recompute exported accounting; no market prices or fills are inferred."""
    def records(name):
        with (Path(directory) / (name + '.csv')).open(newline='', encoding='utf-8') as handle:
            return list(csv.DictReader(handle))
    def key(row):
        return tuple(row[x] for x in ('date', 'accounting', 'segment', 'policy', 'path'))
    def number(row, field):
        value = float(row[field])
        if not math.isfinite(value):
            raise AssertionError('Nonfinite ledger value: ' + field)
        return value
    trades, daily, cash = (records(name) for name in ('trades', 'daily', 'cash_events'))
    pnl, cash_change, episodes = defaultdict(list), defaultdict(list), defaultdict(set)
    max_trade_error = max_daily_error = max_cash_error = 0.0
    resolved = 0
    for row in trades:
        if row['status'] != 'RESOLVED':
            continue
        k = key(row)
        if row['episode_id'] in episodes[k]:
            raise AssertionError('Duplicate resolved episode in one account')
        episodes[k].add(row['episode_id'])
        # exit_price already includes additional stop slippage; never subtract it twice.
        recalculated = (number(row, 'notional') * (number(row, 'exit_price') / number(row, 'entry_price') - 1)
                        - number(row, 'entry_fee') - number(row, 'exit_fee'))
        error = abs(recalculated - number(row, 'net_pnl'))
        max_trade_error = max(max_trade_error, error)
        pnl[k].append(recalculated)
        resolved += 1
    for row in cash:
        cash_change[key(row)].append(number(row, 'cash_delta'))
    matched, seen = defaultdict(dict), set()
    evaluated = 0
    for row in daily:
        k = key(row)
        if k in seen:
            raise AssertionError('Duplicate daily account record')
        seen.add(k)
        if row['status'] != 'EVALUABLE':
            continue
        recorded = number(row, 'net_pnl')
        change = number(row, 'closing_cash') - number(row, 'opening_cash')
        max_daily_error = max(max_daily_error, abs(math.fsum(pnl[k]) - recorded), abs(change - recorded))
        max_cash_error = max(max_cash_error, abs(math.fsum(cash_change[k]) - change))
        matched[k[:3]][k[3:]] = episodes[k]
        evaluated += 1
    if max(max_trade_error, max_daily_error, max_cash_error) > tolerance:
        raise AssertionError('Exported ledger accounting does not reconcile')
    for accounts in matched.values():
        if set(accounts) != set(SCENARIOS):
            raise AssertionError('Evaluable day is missing a policy/path account')
        episode_sets = list(accounts.values())
        if any(items != episode_sets[0] for items in episode_sets[1:]):
            raise AssertionError('Evaluable policy/path accounts do not share the same episodes')
    return dict(resolved_trade_rows=resolved, evaluated_account_days=evaluated,
                matched_six_account_days=len(matched), maximum_trade_error_inr=max_trade_error,
                maximum_daily_error_inr=max_daily_error, maximum_cash_error_inr=max_cash_error,
                tolerance_inr=tolerance, status='PASS',
                scope='Recorded resolved trades and evaluable days only; incomplete days are retained, not repaired')


## 6. Run the experiment

The same seven execution settings and two timestamp interpretations produce fourteen scenario runs. Each contains the same three exit policies under two price paths. These are matched comparisons, not independent experiments.

The code records data, scientific functions, the declared notebook-code bundle, settings, calendar and analysis rules before replay. Local tests verify the code-bundle declaration; runtime checks rehash scientific functions and parsed data. Check the downloaded executed notebook against the reviewed code bundle before freezing. A final run requires explicit approval and an identical completed-validation freeze. The final access record is written first and cannot be silently replaced. The freeze is an internal version record; intervals remain exploratory.

Detailed market ledgers stay in the private runtime. Shared exports contain aggregate results, reconciliation checks and provenance. An economic funding failure or unknown held-position value remains in the record.


In [ ]:
def peer_review_tables(stage, means, episodes, es, statuses, split, sensitivity,
                       capital=100000.0, allocation=.5, stock_count=5, cost_bp=15.0):
    """Keep stages separate; recover gross returns only for the matched fixed-fee model."""
    if stage not in {"pilot", "development", "validation", "final"}:
        raise ValueError("Unknown analysis stage")
    if not (capital > 0 and 0 < allocation <= 1 and stock_count > 0 and cost_bp == 15):
        raise ValueError("Positive capital/allocation and the declared 15 bp reference are required")

    def one(rows, **keys):
        matches = [r for r in rows if all(r.get(k) == v for k, v in keys.items())]
        if len(matches) != 1:
            raise ValueError(f"Expected one row for {keys}; found {len(matches)}")
        return matches[0]

    def equal(a, b):
        if not math.isclose(a, b, rel_tol=0, abs_tol=1e-8):
            raise ValueError(f"Inconsistent matched aggregates: {a} != {b}")

    gross, samples, tails = [], [], []
    fraction = allocation / stock_count
    scheduled = int(one(split, stage=stage)["scheduled_sessions"])
    for label in ("start", "end"):
        count = int(one(episodes, timestamp_label=label)["unique_episodes"])
        if count < 0:
            raise ValueError("Episode count cannot be negative")
        ref_status = [r for r in statuses if r["timestamp_label"] == label and r["setting"] == "reference"]
        primary = [r for r in ref_status if r["accounting"] == "independent"]
        equal(sum(int(r["scheduled_days"]) for r in primary), scheduled)
        eligible = int(one(primary, status="EVALUABLE")["scheduled_days"])
        if eligible <= 0:
            raise ValueError("No primary evaluable days; do not report a gross average")
        funded = sum(int(r["scheduled_days"]) for r in ref_status
                     if r["accounting"] == "funded" and r["status"] == "EVALUABLE")
        samples.append(dict(stage=stage, timestamp_label=label, scheduled_days=scheduled,
            primary_evaluable_days=eligible, primary_unavailable_days=scheduled-eligible,
            explicitly_excluded_days=sum(int(r["scheduled_days"]) for r in primary
                                         if r["status"] == "EXCLUDED_SESSION"),
            incomplete_candidate_days=sum(int(r["scheduled_days"]) for r in primary
                                         if r["status"] == "INCOMPLETE_CANDIDATE_COVERAGE"),
            funded_identified_days=funded, reference_unique_episodes=count,
            mean_episodes_per_primary_day=count/eligible))
        for policy in ("time_only", "stop_only", "stop_and_target"):
            keys = dict(timestamp_label=label, policy=policy, path="O-L-H-C")
            rows = {s: one(means, setting=s, **keys) for s in ("reference", "cost_5", "cost_25")}
            for row in rows.values():
                equal(float(row["evaluable_days"]), eligible)
            net = float(rows["reference"]["mean_daily_pnl_bp"])
            fee = cost_bp * fraction * count / eligible
            # Independent arithmetic check: a 20 bp fee change across identical entries.
            fee_slope = (float(rows["cost_5"]["mean_daily_pnl_bp"])
                         - float(rows["cost_25"]["mean_daily_pnl_bp"])) / 20
            equal(fee_slope, fraction*count/eligible)
            gross_daily = net + fee
            gross.append(dict(stage=stage, timestamp_label=label, policy=policy,
                primary_days=eligible, unique_episodes=count, entry_fraction_of_capital=fraction,
                net_bp_per_day=net, fee_bp_per_day=fee, gross_bp_per_day=gross_daily,
                gross_bp_per_trade=gross_daily*eligible/(fraction*count) if count else None,
                net_inr_per_day=net*capital/10000, gross_inr_per_day=gross_daily*capital/10000,
                reference_round_trip_fee_bp=cost_bp))
            tail = one(es, **keys)
            equal(float(tail["n_days"]), eligible)
            equal(float(tail["tail_mass_days"]), .025*eligible)
            tails.append(dict(stage=stage, **{k: tail[k] for k in (
                "timestamp_label", "policy", "n_days", "tail_mass_days", "ES975_loss_bp",
                "leave_one_day_out_min_bp", "leave_one_day_out_max_bp")}))
    aligned = [dict(stage=stage, **row) for row in sensitivity]
    for row in aligned:
        equal(float(row["scenario_difference_bp"])-float(row["reference_difference_bp"]),
              float(row["change_from_reference_bp"]))
        n = int(float(row["aligned_days"]))
        if n <= 0 or n > scheduled:
            raise ValueError("Invalid aligned sample size")
    return {"gross_net_returns": gross, "sample_accounting": samples,
            "expected_shortfall_sensitivity": tails, "execution_sensitivity": aligned}


# These functions record versions and boundaries; they do not tune the strategy.
M7_CODE_BUNDLE_SHA256 = 'e4b738d4e667a40e3381efe37f1a8b429367a95b50493fb2837da49035f2151d'

def m7_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False,
                                     separators=(',', ':')).encode()).hexdigest()

def m7_check_stage(stage, dates, partitions, final_approved=False):
    if stage not in {'pilot', 'development', 'validation', 'final'}:
        raise RuntimeError('Unknown evaluation stage')
    if not dates or (stage != 'pilot' and dates != partitions[stage]):
        raise RuntimeError('Evaluation dates differ from the declared partition')
    if stage != 'final' and max(dates) >= min(partitions['final']):
        raise RuntimeError('Non-final evaluation cannot enter the final partition')
    if stage == 'final' and not final_approved:
        raise RuntimeError('Final test locked: explicit final-access approval is required')

def m7_science_payload():
    # An explicit function list is stable when later cells are executed again.
    implementation = {name: inspect.getsource(globals()[name]).strip()
                      for name in M7_SCIENTIFIC_FUNCTIONS}
    return dict(protocol_id='NSE-M7-fixed-three-exit-v1', configuration=CFG,
        settings=SETTINGS, inference=INFERENCE, bootstrap_diagnostic=DIAGNOSTIC_SPEC,
        calendar=[d.isoformat() for d in CALENDAR],
        splits=split_table.astype(str).to_dict('records'),
        parsed_data_sha256=parsed_data_fingerprint(DB_PATH),
        implementation_sha256=m7_digest(implementation),
        declared_notebook_code_sha256=M7_CODE_BUNDLE_SHA256,
        declared_source_id=DECLARED_SOURCE_ID,
        source_reverification=DATA_AUDIT['raw_inputs_reverified'],
        analysis='Original matched primary comparison; gross/net fee decomposition, tail deletion and aligned sensitivity diagnostics; no tuning or pooling of stages.',
        inference_interpretation='All nominal intervals are exploratory; no validated confirmatory amendment is claimed.',
        timestamp_semantics='unconfirmed: retain both conditional interpretations',
        source_precedence='original monthly delivery, except declared amendment M7-A1-2023-06-23 (complete supplementary session for 23 June 2023)',
        source_amendments=SOURCE_AMENDMENTS)

def m7_freeze_validation(path, executed_science, current_science, manifest, checks):
    """Freeze only a completed, unchanged, raw-data validation with all ledger checks."""
    if (manifest.get('stage') != 'validation' or not manifest.get('raw_inputs_reverified')
            or manifest.get('independent_economic_failure') or not manifest.get('completed_utc')):
        raise RuntimeError('Freeze requires completed raw-data validation without independent funding failure')
    expected = {(label, setting['name']) for label in ('start', 'end')
                for setting in executed_science['settings']}
    observed = {(row.get('timestamp_label'), row.get('setting')) for row in checks}
    if len(checks) != 14 or observed != expected or any(row.get('status') != 'PASS' for row in checks):
        raise RuntimeError('Freeze requires all fourteen ledger reconciliations to pass')
    digest = m7_digest(executed_science)
    if current_science != executed_science or manifest.get('protocol_sha256') != digest:
        raise RuntimeError('Code, data or settings changed after validation; do not freeze stale results')
    freeze = dict(protocol_sha256=digest, science=executed_science,
        validation_run_id=manifest['run_id'], created_utc=datetime.now(timezone.utc).isoformat(),
        validation_reviewed=True, ledger_reconciliations_passed=14,
        note='Internal version freeze after validation review; not independent preregistration or proof of statistical coverage')
    path = Path(path)
    if path.exists():
        previous = json.loads(path.read_text(encoding='utf-8'))
        if previous.get('protocol_sha256') != digest or previous.get('science') != executed_science:
            raise RuntimeError('Existing freeze differs; preserve it and document an amendment')
        return previous
    with path.open('x', encoding='utf-8') as handle:
        json.dump(freeze, handle, indent=2, allow_nan=False)
    return freeze

def m7_record_final_access(freeze_path, science, approved, existing_database,
                          access_path, run_id, final_dates):
    """Check the reviewed freeze and create an exclusive record before final replay."""
    if not approved:
        raise RuntimeError('Final test locked: explicit final-access approval is required')
    if existing_database or not science.get('source_reverification'):
        raise RuntimeError('Final test locked: original inputs must be reverified')
    if not Path(freeze_path).is_file():
        raise RuntimeError('Final test locked: completed-validation freeze is missing')
    frozen = json.loads(Path(freeze_path).read_text(encoding='utf-8'))
    digest = m7_digest(science)
    if (frozen.get('protocol_sha256') != digest or frozen.get('science') != science
            or not frozen.get('validation_reviewed')
            or frozen.get('ledger_reconciliations_passed') != 14):
        raise RuntimeError('Final test locked: data, code or settings differ from the reviewed freeze')
    record = dict(run_id=run_id, protocol_sha256=digest,
        validation_run_id=frozen['validation_run_id'],
        access_utc=datetime.now(timezone.utc).isoformat(),
        final_start=str(final_dates[0]), final_end=str(final_dates[-1]),
        scheduled_sessions=len(final_dates), explicit_approval=True,
        status='Access recorded before replay; completion is recorded in the run manifest')
    try:
        with Path(access_path).open('x', encoding='utf-8') as handle:
            json.dump(record, handle, indent=2)
    except FileExistsError:
        raise RuntimeError('Final access already recorded; preserve the record and review any recovery before rerunning') from None
    return record

def m7_conclusions(stage, gross, intervals, mechanisms, economic_failure):
    """Report observed economics first, followed by the fixed primary comparison."""
    rows = gross.to_dict('records')
    reference = [r for r in rows if r['policy'] == 'time_only']
    lines = [f'**Results — {stage}.**']
    if reference:
        low, high = min(r['gross_bp_per_day'] for r in reference), max(r['gross_bp_per_day'] for r in reference)
        fee_low, fee_high = min(r['fee_bp_per_day'] for r in reference), max(r['fee_bp_per_day'] for r in reference)
        lines.append(f'Time-only gross returns were {low:.4f}–{high:.4f} bp per evaluable day, against assumed fees of {fee_low:.4f}–{fee_high:.4f} bp per day.')
        if all(r['net_bp_per_day'] < 0 for r in rows):
            lines.append('All three exit policies had negative net mean returns under both timestamp interpretations at reference execution.')
        else:
            positive = [f"{r['timestamp_label']} / {r['policy']}" for r in rows if r['net_bp_per_day'] > 0]
            lines.append('Positive net sample means: ' + (', '.join(positive) if positive else 'none') + '.')
    for label in ('start', 'end'):
        found = intervals.query("timestamp_label == @label and path == 'O-L-H-C' and contrast == 'combined_minus_time' and block_length == 5") if not intervals.empty else pd.DataFrame()
        if found.empty:
            lines.append(f'Under {label} labels, no primary estimate is available.')
            continue
        row = found.iloc[0]
        text = f'Combined minus time-only was {row.mean_difference_bp:.4f} bp per evaluable day under {label} labels (n={int(row.n_days)}).'
        if pd.notna(row.lower_bp):
            text += f' The exploratory nominal 95% interval was [{row.lower_bp:.4f}, {row.upper_bp:.4f}] bp.'
        else:
            text += ' The interval was unavailable under the declared support rules.'
        lines.append(text)
    if not mechanisms.empty:
        lines.append('Ambiguous reference episodes: ' + ', '.join(
            f'{r.timestamp_label} {int(r.ambiguous_episodes)}/{int(r.unique_episodes)}'
            for r in mechanisms.itertuples()) + '.')
    lines.append('**Limitations.** These sample estimates condition on common evaluable sessions. Costs and fills are scenarios. All nominal intervals remain exploratory because coverage is not validated. ES has a sparse tail; funded drawdown uses separately identified account paths. Two price paths do not recover actual fills or establish general path robustness.')
    if economic_failure:
        lines.append('An independent-session funding failure prevents primary superiority inference; surviving-day estimates cannot replace the planned comparison.')
    lines.append('Final outcomes were evaluated under the recorded protocol.' if stage == 'final'
                 else 'The final partition was not evaluated in this run.')
    return '\n\n'.join(lines)


In [ ]:
def parsed_data_fingerprint(db_path):
    digest = hashlib.sha256()
    with sqlite3.connect(Path(db_path).resolve().as_uri()+'?mode=ro',uri=True) as con:
        for row in con.execute('SELECT symbol,stamp,o,h,l,c,volume FROM bars ORDER BY symbol,stamp'):
            digest.update((json.dumps(row,separators=(',',':'),allow_nan=False)+'\n').encode())
    return digest.hexdigest()

def read_private_csv(path):
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

def run_one_scenario(label, setting):
    scenario = f'{label}__{setting["name"]}'
    private = WORK_DIR / scenario
    private.mkdir()
    equity_state = {'writer':None}
    started = clock.perf_counter()
    with (private/'equity.csv').open('w',newline='',encoding='utf-8') as handle:
        def sink(row):
            if equity_state['writer'] is None:
                equity_state['writer'] = csv.DictWriter(handle,fieldnames=list(row))
                equity_state['writer'].writeheader()
            equity_state['writer'].writerow(row)
        result = run_research(DB_PATH,{**CFG,**setting,'timestamp_label':label},CALENDAR,EVALUATION_DATES,equity_sink=sink)
    for name in ['daily','trades','cash_events','diagnostics','candidates','segments']:
        pd.DataFrame(result[name]).to_csv(private/f'{name}.csv',index=False)
    result['metadata']['elapsed_seconds'] = clock.perf_counter()-started
    (private/'metadata.json').write_text(json.dumps(result['metadata'],indent=2)+'\n')
    # Only small daily/diagnostic tables remain in memory across scenarios.
    return {'daily':pd.DataFrame(result['daily']), 'diagnostics':pd.DataFrame(result['diagnostics']),
            'segments':pd.DataFrame(result['segments']), 'metadata':result['metadata'], 'private_dir':private}

DECLARED_SOURCE_ID = 'NSE-M7-2026-11-10-v1'
M7_SCIENTIFIC_FUNCTIONS = ('month_keys', 'file_hash', 'archive_command', 'archive_lines', 'member_named', 'archive_members', 'vendor_rows', 'discover_sources', 'build_database', 'supplement_rows', 'fold_preopen', 'apply_source_amendments', 'source_status_markdown', 'calendar_date', 'research_parameters', 'session_bounds', 'read_session', 'scheduled_decisions', 'window_feature', 'plan_session', 'fresh_accounts', 'barrier_fill', 'release_cash', 'available_cash', 'close_position', 'replay_session', 'run_research', 'run_notebook_method_checks', 'expected_shortfall', 'eligible_segments', 'paired_block_interval', 'tail_rows', 'diagnostic_mean_weights', 'bootstrap_reliability_diagnostic', 'reconcile_ledgers', 'peer_review_tables', 'm7_digest', 'm7_check_stage', 'm7_science_payload', 'm7_freeze_validation', 'm7_record_final_access', 'm7_conclusions', 'parsed_data_fingerprint', 'read_private_csv', 'run_one_scenario')
SCIENCE_PAYLOAD = m7_science_payload()
PROTOCOL_SHA256 = m7_digest(SCIENCE_PAYLOAD)
FREEZE_PATH = SAVE_BASE / 'm7_protocol_freeze.json'
FINAL_ACCESS_PATH = SAVE_BASE / 'm7_final_access.json'
FINAL_ACCESS_RECORD = None
m7_check_stage(RUN_STAGE, EVALUATION_DATES, PARTITIONS, FINAL_ACCESS_APPROVED)
if RUN_STAGE == 'final':
    FINAL_ACCESS_RECORD = m7_record_final_access(FREEZE_PATH, SCIENCE_PAYLOAD,
        FINAL_ACCESS_APPROVED, EXISTING_DB_PATH, FINAL_ACCESS_PATH, RUN_ID, EVALUATION_DATES)
    print('Matching validation freeze verified; final access is recorded before replay.')
else:
    print('Final-test outcomes are not evaluated in this run.')
print('Protocol SHA-256:', PROTOCOL_SHA256)
RUNS = {}
for label in ['start','end']:
    for setting in SETTINGS:
        RUNS[(label,setting['name'])] = run_one_scenario(label,setting)
        current = RUNS[(label,setting['name'])]
        common = current['daily'].query("accounting == 'independent' and status == 'EVALUABLE'").date.nunique()
        print(f'{label:5} {setting["name"]:18} | {common:4} evaluable days | {current["metadata"]["elapsed_seconds"]:.1f}s')


## 7. Sample and exclusions

Maintain all the scheduled dates in the ledger. Valid no-signal days count as 0, and unavailable / failed days do not count as 0. Known missing inputs before entry permit funded abstention, which may result in a discrepancy between the funded and complete case counts.

Eligible dates for all six primary policy/path series must be the same. Count failures before summarising. The displayed candidate illustrates an observed signal, not an additional independent observation.


In [ ]:
ALL_DAILY = pd.concat([run['daily'].assign(timestamp_label=label,setting=name)
                      for (label,name),run in RUNS.items()],ignore_index=True)
ALL_DAILY['return_bp'] = ALL_DAILY.return_on_reference_capital*10000
status_table = (ALL_DAILY.groupby(['timestamp_label','setting','accounting','status'])
                .date.nunique().rename('scheduled_days').reset_index())
display(status_table[status_table.setting.eq('reference')])
ECONOMIC_FAILURE = bool(((ALL_DAILY.accounting=='independent') & ALL_DAILY.status.eq('FUNDING_FAILURE')).any())
print('Independent-session economic failure:',ECONOMIC_FAILURE)
for key, run in RUNS.items():
    good = run['daily'].query("accounting == 'independent' and status == 'EVALUABLE'")
    sets = [set(g.date) for _,g in good.groupby(['policy','path'])]
    assert not sets or (len(sets)==6 and all(s==sets[0] for s in sets))
candidate_example = read_private_csv(RUNS[('start','reference')]['private_dir']/'candidates.csv')
if not candidate_example.empty:
    first = candidate_example.iloc[0]
    assert np.isclose(first.z,(first.relative_return-first.prior_relative_mean)/first.prior_relative_sd)
    day_position = CALENDAR.index(date.fromisoformat(first.date))
    print('First recorded signal formula verified; the worked public example above uses synthetic values.')
else:
    print('No eligible market signal in this scenario; do not invent a worked market example.')


### Independent accounting check

Recalculate P&L from the exported entry/exit prices and fees, match trade totals to daily accounts, reconcile cash movements, and check that evaluable policy/path accounts contain the same episodes. Any discrepancy stops the notebook. A passing ledger check establishes accounting consistency, not real execution or profitability.


In [ ]:
LEDGER_RECONCILIATION = pd.DataFrame([
    dict(timestamp_label=label, setting=name, **reconcile_ledgers(run['private_dir']))
    for (label, name), run in RUNS.items()])
display(LEDGER_RECONCILIATION)


## 8. Returns and exit mechanisms

Means are daily P&L / fixed capital in basis points, and not returns on compounded wealth. For each date, effects pair that date. Primary contrast is combined minus time; stop only is secondary.

Interpret the differences in policy using exit reasons and the number of unique both-hit episodes. Exit durations are ranges, not seconds, and they are within minutes. Zero ambiguity is a result, NOT a reason for barrier retuning.


In [ ]:
MEANS = (ALL_DAILY.query("accounting == 'independent' and status == 'EVALUABLE'")
         .groupby(['timestamp_label','setting','policy','path']).return_bp.agg(['count','mean','std']).reset_index()
         .rename(columns={'count':'evaluable_days','mean':'mean_daily_pnl_bp','std':'daily_sd_bp'}))
CONTRASTS = [('combined_minus_time','stop_and_target','time_only'),
             ('stop_minus_time','stop_only','time_only'),('combined_minus_stop','stop_and_target','stop_only')]
PAIRED = []
for (label,name), run in RUNS.items():
    good = run['daily'].query("accounting == 'independent' and status == 'EVALUABLE'").copy()
    good['bp'] = good.return_on_reference_capital*10000
    for path,group in good.groupby('path'):
        wide = group.pivot(index='date',columns='policy',values='bp').sort_index()
        assert wide.notna().all().all()
        for contrast,left,right in CONTRASTS:
            PAIRED.append(dict(timestamp_label=label,setting=name,path=path,contrast=contrast,n_days=len(wide),
                               mean_difference_bp=(wide[left]-wide[right]).mean()))
PAIRED = pd.DataFrame(PAIRED,columns=['timestamp_label','setting','path','contrast','n_days','mean_difference_bp'])
display(MEANS[MEANS.setting.eq('reference')].round(4))
display(PAIRED[PAIRED.setting.eq('reference')].round(4))
mechanism_records = []
for label in ['start','end']:
    trades = read_private_csv(RUNS[(label,'reference')]['private_dir']/'trades.csv')
    if trades.empty:
        mechanism_records.append(dict(timestamp_label=label,unique_episodes=0,ambiguous_episodes=0,ambiguity_rate=np.nan))
        continue
    kept = trades[(trades.accounting=='independent') & trades.status.eq('RESOLVED') & trades.included_primary_day.eq(True)]
    if kept.empty:
        print(label, ': no resolved episodes on complete-case dates; ambiguity rate unavailable.')
        mechanism_records.append(dict(timestamp_label=label,unique_episodes=0,ambiguous_episodes=0,ambiguity_rate=np.nan))
        continue
    counts = kept.groupby(['policy','path','exit_reason']).size().rename('executions').reset_index()
    display(counts.assign(timestamp_label=label))
    both = kept[kept.ambiguous.eq(True)].episode_id.nunique()
    unique = kept.episode_id.nunique()
    mechanism_records.append(dict(timestamp_label=label,unique_episodes=unique,ambiguous_episodes=both,
                                   ambiguity_rate=both/unique if unique else np.nan))
MECHANISMS = pd.DataFrame(mechanism_records,columns=['timestamp_label','unique_episodes','ambiguous_episodes','ambiguity_rate'])
display(MECHANISMS)
fig, ax = plt.subplots(figsize=(8,3.5))
ref_plot = PAIRED.query("setting == 'reference' and path == 'O-L-H-C'")
if not ref_plot.empty:
    ref_plot.pivot(index='contrast',columns='timestamp_label',values='mean_difference_bp').rename(index={
        'combined_minus_stop':'Combined − stop','combined_minus_time':'Combined − time',
        'stop_minus_time':'Stop − time'}).plot.bar(ax=ax,rot=0)
    ax.legend(title='Minute labels')
ax.axhline(0,color='black',lw=.8)
ax.set(ylabel='Paired daily P&L difference\n(bp of reference capital)',xlabel='Exit-policy contrast',
       title=f'{RUN_STAGE}: reference execution, O-L-H-C')
ax.tick_params(axis='x',labelsize=8); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'paired_effects.png'); plt.show()


## 9. Funded equity and drawdown

Funded accounts carry realised P&L, fixed notionals and fees are applied and value shares are at minute closes. Early proceeds enter equity and kept on hold until the scheduled expiry. Losses or unknowns are not repaired by a deposit; each partition begins with C₀.

For $DD_t=1-E_t/\max_{u\le t}E_u$:

| Output | Sampling rule |
|---|---|
| `max_drawdown_pct` | All the minute-close and known-flat boundary marks are kept; consecutive marks that do not change may be removed |
| `daily_close_max_drawdown_pct` — | identified closing cash for each completed session, with first capital as the starting peak cash |

No interrupted day marks are daily closes. If a segment has no daily-close identified, then there is no daily-close statistic. Segments are never connected. For each segment, output the date of its last daily close and the number of closes, the last daily close can be before its intraday endpoint. Only the graph will be downsampled. Neither measure reveals within-minute losses and if a funded prefix is terminated then that does not replace the primary estimate.


In [ ]:
FUNDED_RISK, PLOT_EQUITY = [], []
for label in ['start','end']:
    path = RUNS[(label,'reference')]['private_dir']/'equity.csv'
    maxima, samples, counters = {}, {}, {}
    if path.stat().st_size == 0:
        continue
    for chunk in pd.read_csv(path,chunksize=25000):
        for key, group in chunk.groupby(['segment','policy','path'],sort=False):
            old = maxima.setdefault(key,dict(max_drawdown=0.0,marks=0,first_date=str(group.date.iloc[0]),last_date=None))
            old['max_drawdown'] = max(old['max_drawdown'],float(group.drawdown_fraction.max()))
            old['marks'] += len(group); old['last_date'] = str(group.date.iloc[-1])
            count = counters.get(key,0)
            display_rows = group.iloc[np.where((np.arange(len(group))+count)%25==0)[0]]
            # Always retain the last mark of this chunk; rendering only, not the risk computation.
            display_rows = pd.concat([display_rows,group.tail(1)]).drop_duplicates('timestamp')
            samples.setdefault(key,[]).append(display_rows[['timestamp','equity','drawdown_fraction']])
            counters[key] = count+len(group)
    # Completed funded sessions end flat: their identified closing_cash is exact EOD equity.
    # Never turn the final partial mark of an interrupted day into a daily close.
    run = RUNS[(label,'reference')]
    capitals = run['segments'].set_index('segment').initial_capital.to_dict()
    daily = run['daily'].query("accounting == 'funded'")
    for key,value in maxima.items():
        rows = daily[(daily.segment==key[0]) & (daily.policy==key[1]) & (daily.path==key[2])].sort_values('date')
        known = rows.status.eq('EVALUABLE')
        if (known & (~known).cummax()).any():
            raise ValueError('Identified daily closes cannot resume after a failure within the same funded segment')
        closes = rows.loc[known,'closing_cash'].to_numpy(dtype=float)
        capital = float(capitals[key[0]])
        if not np.isfinite(closes).all() or not np.isfinite(capital) or capital<=0:
            raise ValueError('Daily-close drawdown requires identified closes and positive initial capital')
        levels = np.r_[capital,closes]
        daily_mdd = float(np.max(1-levels/np.maximum.accumulate(levels))) if len(closes) else np.nan
        FUNDED_RISK.append(dict(timestamp_label=label,segment=key[0],policy=key[1],path=key[2],
                                max_drawdown_pct=100*value['max_drawdown'],saved_marks=value['marks'],
                                first_date=value['first_date'],last_date=value['last_date'],
                                daily_close_max_drawdown_pct=100*daily_mdd,
                                identified_daily_closes=len(closes),
                                last_identified_daily_close=rows.loc[known,'date'].iloc[-1] if len(closes) else None,
                                segment_initial_capital=capital))
        sample = pd.concat(samples[key]).drop_duplicates('timestamp').sort_values('timestamp')
        PLOT_EQUITY.append((label,key,sample))
FUNDED_RISK = pd.DataFrame(FUNDED_RISK)
display(FUNDED_RISK.round(4))
fig,axes = plt.subplots(2,2,figsize=(12,6),sharex='col')
for label,key,frame in PLOT_EQUITY:
    if key[2]!='O-L-H-C':
        continue
    j = ['start','end'].index(label)
    ts = pd.to_datetime(frame.timestamp,utc=True)
    axes[0,j].plot(ts,frame.equity,label=f'{key[1].replace("_"," ")} · segment {key[0]}')
    axes[1,j].plot(ts,100*frame.drawdown_fraction)
for j,label in enumerate(['start','end']):
    axes[0,j].set(title=f'{label} labels · funded reference O-L-H-C',ylabel='Marked equity (INR)')
    if axes[0,j].lines: axes[0,j].legend(fontsize=7)
    axes[1,j].set(ylabel='Sampled drawdown (%)',xlabel='Time (UTC; source assumed IST)')
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'funded_equity_drawdown.png'); plt.show()
for label in ['start','end']:
    display(RUNS[(label,'reference')]['segments'].assign(timestamp_label=label))


## 10. Expected shortfall and exploratory intervals

ES adopts the same complete-case independent-session dates as the primary means. Loss = −P&L / C₀; ES can be negative if even the worst sampled days profit. Tied tail losses are not unique extreme day attributions.

Even when they do not include zero, all nominal 95% intervals are exploratory. The diagnostics for serial-dependence indicate potential undercoverage. Confirmatory inference requires a separately documented, validated amendment prior to final-test access. Intervals can't repair missingness or uncertain executions - an interval that contains 0, including an endpoint, is inconclusive.

### Bootstrap reliability diagnostic

Specified after development, this is a diagnostic test of the five-day, 2,000-resample procedure (unchanged from development) prior to validation or final outcomes:

The paired differences are Gaussian with mean 0, stationary SD 1 bp, independence or AR(1) coefficients 0.3 and 0.7. - 200 histories per case which evolve along each development day scheduled and are used to apply both observed eligibility masks. The returns on the market are not included in the simulation. Coverage includes zero endpoints. The rejection is the complement of the coverage. If any intervals were unavailable, report them separately. Coverage MCSE is in percentage points, width and its MCSE in bp. Rates change in 0.5 point increments. - Oracle width with known synthetic covariance. Cached resampling weights are checked against the original interval calculation. MCSE conditions on fixed bootstrap seed and masks.

These Gaussian checks fail to ensure coverage under actual market dynamics, heavy tails, regime changes and outcome driven missingness. They do not choose the block size, modify the trading rules or unlock unsupported intervals.

This diagnostic runs only in the development stage and is skipped during validation.


In [ ]:
ES_ROWS, TAILS, INTERVALS = [], [], []
for label in ['start','end']:
    good = ALL_DAILY.query("timestamp_label == @label and setting == 'reference' and accounting == 'independent' and status == 'EVALUABLE'")
    for path,group in good.groupby('path'):
        wide = group.pivot(index='date',columns='policy',values='return_bp').sort_index()
        for policy in POLICIES:
            values = wide[policy]
            estimate = expected_shortfall(-values)
            leave_one = [expected_shortfall(-values.drop(index=d)) for d in values.index] if len(values)>1 else [np.nan]
            ES_ROWS.append(dict(timestamp_label=label,path=path,policy=policy,n_days=len(values),
                                tail_mass_days=.025*len(values),ES975_loss_bp=estimate,
                                leave_one_day_out_min_bp=np.nanmin(leave_one),leave_one_day_out_max_bp=np.nanmax(leave_one)))
            TAILS.append(tail_rows(values).assign(timestamp_label=label,path=path,policy=policy))
        for contrast,left,right in CONTRASTS:
            for b in INFERENCE['block_lengths']:
                interval = paired_block_interval(wide,[d.isoformat() for d in EVALUATION_DATES],left,right,b,INFERENCE)
                if ECONOMIC_FAILURE:
                    interval.update(status='UNAVAILABLE: independent-account funding failure',lower_bp=np.nan,upper_bp=np.nan)
                INTERVALS.append(dict(timestamp_label=label,path=path,contrast=contrast,**interval))
ES_TABLE = pd.DataFrame(ES_ROWS)
TAIL_TABLE = pd.concat(TAILS,ignore_index=True) if TAILS else pd.DataFrame()
INTERVAL_TABLE = pd.DataFrame(INTERVALS)
display(ES_TABLE.query("path == 'O-L-H-C'").round(4) if not ES_TABLE.empty else ES_TABLE)
display(TAIL_TABLE.head(18))
display(INTERVAL_TABLE.query("path == 'O-L-H-C' and contrast == 'combined_minus_time'") if not INTERVAL_TABLE.empty else INTERVAL_TABLE)
if not ES_TABLE.empty:
    es_wide = ES_TABLE.pivot(index=['timestamp_label','path'],columns='policy',values='ES975_loss_bp')
    es_wide['combined_minus_time_ES_bp'] = es_wide.stop_and_target-es_wide.time_only
    es_wide['stop_minus_time_ES_bp'] = es_wide.stop_only-es_wide.time_only
    display(es_wide.round(4))


BOOTSTRAP_RELIABILITY = pd.DataFrame()
BOOTSTRAP_RELIABILITY_AUDIT = dict(status='Not run outside the development stage')
if RUN_STAGE == 'development':
    if EVALUATION_DATES != PARTITIONS['development']:
        raise RuntimeError('Diagnostics require the complete predeclared development calendar')
    diagnostic_started = clock.perf_counter()
    BOOTSTRAP_RELIABILITY, BOOTSTRAP_RELIABILITY_AUDIT = bootstrap_reliability_diagnostic(
        ALL_DAILY, PARTITIONS['development'], INFERENCE, DIAGNOSTIC_SPEC)
    BOOTSTRAP_RELIABILITY_AUDIT['elapsed_seconds'] = clock.perf_counter()-diagnostic_started
    BOOTSTRAP_RELIABILITY_AUDIT['independent_economic_failure'] = bool(ECONOMIC_FAILURE)
    BOOTSTRAP_RELIABILITY.to_csv(OUTPUT_DIR/'bootstrap_reliability.csv', index=False)
    (OUTPUT_DIR/'bootstrap_reliability_audit.json').write_text(
        json.dumps(BOOTSTRAP_RELIABILITY_AUDIT, indent=2, allow_nan=False)+'\n', encoding='utf-8')
    display(BOOTSTRAP_RELIABILITY[['timestamp_label','phi','eligible_days','replications',
        'coverage_pct','coverage_mcse_pp','rejection_pct','rejection_mcse_pp',
        'mean_width_bp','width_mcse_bp','gaussian_oracle_width_bp',
        'mean_width_over_oracle','unavailable_count']].round(4))
    print('Synthetic diagnostics only; no market inference is certified by this table.')
    print('Elapsed seconds:', round(BOOTSTRAP_RELIABILITY_AUDIT['elapsed_seconds'],2))
else:
    print('Synthetic coverage diagnostics use development masks only; this stage is skipped.')


## 11. Sensitivity of execution and samples

Implement changes to costs, stop penalties and delays, one at a time. Fixed-entry-notional round-trip fees cancel in completed paired contrasts but change absolute returns; actual exit-value fees need not cancel. Stop penalties only apply to stopped trades. Delays shift the entry and reanchor both barriers.

Common dates are used in the main tables for each scenario. Aligned table intersects reference and stress dates to show sample effects. Signals can even change under different timestamp interpretations. The convention and the scenario costs are not selected based on returns and not even presented as verified historical execution.


In [ ]:
ALIGNMENT = []
for label in ['start','end']:
    part = ALL_DAILY.query("timestamp_label == @label and accounting == 'independent' and status == 'EVALUABLE' and path == 'O-L-H-C'")
    ref = part[part.setting.eq('reference')].pivot(index='date',columns='policy',values='return_bp')
    for setting in SETTINGS:
        current = part[part.setting.eq(setting['name'])].pivot(index='date',columns='policy',values='return_bp')
        days = sorted(set(ref.index)&set(current.index))
        if not days:
            continue
        for contrast,left,right in CONTRASTS:
            baseline = (ref.loc[days,left]-ref.loc[days,right]).mean()
            stressed = (current.loc[days,left]-current.loc[days,right]).mean()
            ALIGNMENT.append(dict(timestamp_label=label,setting=setting['name'],contrast=contrast,
                                  aligned_days=len(days),reference_difference_bp=baseline,
                                  scenario_difference_bp=stressed,change_from_reference_bp=stressed-baseline))
ALIGNMENT = pd.DataFrame(ALIGNMENT)
display(ALIGNMENT[ALIGNMENT.contrast.eq('combined_minus_time')].round(4) if not ALIGNMENT.empty else ALIGNMENT)
for name in ['cost_5','cost_25']:
    aligned = ALIGNMENT[ALIGNMENT.setting.eq(name)] if not ALIGNMENT.empty else ALIGNMENT
    if not aligned.empty:
        assert np.allclose(aligned.change_from_reference_bp,0,atol=1e-8)
fig,ax = plt.subplots(figsize=(9,3.4))
if not ALIGNMENT.empty:
    ALIGNMENT[ALIGNMENT.contrast.eq('combined_minus_time')].pivot(index='setting',columns='timestamp_label',values='scenario_difference_bp').plot.bar(ax=ax,rot=25)
    ax.legend(title='Minute labels')
ax.axhline(0,color='black',lw=.8); ax.set(ylabel='Combined − time mean effect\n(bp of reference capital)',xlabel='Declared execution scenario',
                                       title=f'{RUN_STAGE}: one-at-a-time sensitivity, O-L-H-C')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'execution_sensitivity.png'); plt.show()


### Gross returns, sample counts and tail sensitivity

The four diagnostic tables use the completed current-stage aggregates. They separate gross returns from assumed fees, scheduled from evaluable and funded dates, aligned execution sensitivities, and ES leave-one-day-out ranges. They do not add trades or tune the strategy. Read the gross/fee/net table before interpreting the exit contrasts.


In [ ]:
PEER_REVIEW_TABLES = {name: pd.DataFrame(rows) for name, rows in peer_review_tables(
    RUN_STAGE, MEANS.to_dict('records'), MECHANISMS.to_dict('records'),
    ES_TABLE.to_dict('records'), status_table.to_dict('records'),
    [dict(stage=RUN_STAGE, scheduled_sessions=len(EVALUATION_DATES))],
    ALIGNMENT.to_dict('records'), capital=CFG['reference_capital_inr'],
    allocation=CFG['allocation_fraction'], stock_count=len(CFG['universe'])).items()}
for name, table in PEER_REVIEW_TABLES.items():
    print(name.replace('_', ' ').title())
    display(table.round(4))
print('Gross means remove assumed fixed fees; they do not establish an edge over random entries.')
print('ES leave-one-day-out ranges are descriptive, not confidence intervals or common-date policy rankings.')


## 12. Results and limitations

Read the gross/fee/net decomposition first, then the matched exit-policy comparison and risk measures. The generated text reports the current stage only. Keep development, validation and final results separate in the report. Detailed limitations are collected after the results rather than repeated after every number.


In [ ]:
CONCLUSION = m7_conclusions(RUN_STAGE, PEER_REVIEW_TABLES['gross_net_returns'],
                            INTERVAL_TABLE, MECHANISMS, ECONOMIC_FAILURE)
display(Markdown(CONCLUSION))
if not BOOTSTRAP_RELIABILITY.empty:
    display(BOOTSTRAP_RELIABILITY.round(4))


## 13. Save the run and review the freeze

The first execution saves a new aggregate-results folder and ZIP. After reviewing the completed validation and its fourteen reconciliations, set `READY_TO_FREEZE=True` below and rerun only this cell. The function checks that code, settings and inputs still match the executed validation. An incompatible existing freeze is never overwritten.

Before a final run, reset this checkbox, start a fresh runtime and set the final-access controls at the top. A final run exports its frozen protocol and access record. Save the executed notebook separately; review outputs for private paths or source observations before sharing. Raw prices and private ledgers are not included in the results ZIP.


In [ ]:
READY_TO_FREEZE = False # @param {type:"boolean"}
# Only set this after reviewing the completed validation, then rerun this cell alone.
import base64, html, zipfile

tables = {'calendar_split':split_table,'coverage_monthly':coverage_table,'coverage_daily':coverage_daily,
          'status_counts':status_table,'policy_means':MEANS,'paired_effects':PAIRED,
          'exit_ambiguity':MECHANISMS,'funded_drawdown':FUNDED_RISK,'expected_shortfall':ES_TABLE,
          'tail_dates':TAIL_TABLE,'intervals':INTERVAL_TABLE,'aligned_sensitivity':ALIGNMENT,'bootstrap_reliability':BOOTSTRAP_RELIABILITY,'ledger_reconciliation':LEDGER_RECONCILIATION}
tables.update(PEER_REVIEW_TABLES)
for name,table in tables.items():
    table.to_csv(OUTPUT_DIR/f'{name}.csv',index=False)
from importlib.metadata import PackageNotFoundError, version
runtime_packages = {}
for package in ('numpy', 'pandas', 'matplotlib', 'ipython', 'nbformat', 'nbclient', 'ipykernel', 'nbconvert'):
    try:
        runtime_packages[package] = version(package)
    except PackageNotFoundError:
        runtime_packages[package] = 'not installed'
manifest = dict(run_id=RUN_ID,stage=RUN_STAGE,completed_utc=datetime.now(timezone.utc).isoformat(),
                protocol_sha256=PROTOCOL_SHA256,science=SCIENCE_PAYLOAD,
                python=sys.version,platform=platform.platform(),packages=runtime_packages,
                declared_source_id=DECLARED_SOURCE_ID, final_test_enabled=RUN_STAGE == 'final',
                final_test_evaluated=RUN_STAGE == 'final', protocol_freezing_enabled=True,
                notebook_code_sha256=M7_CODE_BUNDLE_SHA256,
                final_access_record=FINAL_ACCESS_RECORD,
                independent_economic_failure=ECONOMIC_FAILURE,
                post_review_diagnostics='Original aggregate diagnostics; M7 release adds reviewed validation/final controls without retuning',
                source_status=DATA_AUDIT['mode'],raw_inputs_reverified=DATA_AUDIT['raw_inputs_reverified'],
                source_files=len(DATA_AUDIT.get('input_files',[])),
                caveat='A locally verified cached pilot does not establish a successful mounted-Drive/full-study run.')
try:
    import resource
    manifest['process_peak_rss_mib'] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/(1024 if sys.platform!='darwin' else 1024**2)
except ImportError:
    manifest['process_peak_rss_mib'] = None
(OUTPUT_DIR/'run_manifest.json').write_text(json.dumps(manifest,indent=2,default=str)+'\n',encoding='utf-8')
(OUTPUT_DIR/'conclusions.md').write_text(CONCLUSION+'\n',encoding='utf-8')
# Source provenance retains checksums/member names without account-specific root paths.
public_audit = json.loads(json.dumps(DATA_AUDIT))
for item in public_audit.get('input_files',[]):
    if 'source' in item: item['source'] = Path(item['source']).name
public_audit['excluded_supplements'] = [Path(p).name for p in public_audit.get('excluded_supplements',[])]
(OUTPUT_DIR/'input_audit.json').write_text(json.dumps(public_audit,indent=2)+'\n',encoding='utf-8')
parts = ['<!doctype html><meta charset="utf-8"><title>NSE research run</title>',
         '<style>body{font:16px Georgia;max-width:1100px;margin:40px auto;padding:0 20px;color:#20252c}table{border-collapse:collapse;font:12px Arial;width:100%;margin:20px 0}th,td{padding:6px;border-bottom:1px solid #ddd;text-align:right}img{max-width:100%}pre{white-space:pre-wrap}h1,h2{font-family:Arial}</style>',
         '<h1>'+html.escape(CFG['study_title'])+'</h1>', '<p>Group 17838 · '+html.escape(RUN_ID)+' · '+RUN_STAGE+'</p>',
         '<h2>Evidence and interpretation</h2><pre>'+html.escape(CONCLUSION)+'</pre>']
for name in ['policy_means','paired_effects','status_counts','exit_ambiguity','expected_shortfall','funded_drawdown','intervals','bootstrap_reliability','ledger_reconciliation',*PEER_REVIEW_TABLES]:
    parts.extend(['<h2>'+name.replace('_',' ').title()+'</h2>',tables[name].to_html(index=False,float_format=lambda x:f'{x:.5f}')])
for path in sorted(OUTPUT_DIR.glob('*.png')):
    parts.append('<h2>'+path.stem.replace('_',' ').title()+'</h2><img alt="'+html.escape(path.stem)+'" src="data:image/png;base64,'+base64.b64encode(path.read_bytes()).decode()+'">')
parts.append('<p>Protocol SHA-256: '+PROTOCOL_SHA256+'</p><p>Read the notebook for assumptions, equations, tests, limitations and cited references. No course submission is performed.</p>')
(OUTPUT_DIR/'research_report.html').write_text('\n'.join(parts),encoding='utf-8')
if READY_TO_FREEZE and RUN_STAGE == 'validation':
    m7_check_stage(RUN_STAGE, EVALUATION_DATES, PARTITIONS, FINAL_ACCESS_APPROVED)
    frozen = m7_freeze_validation(FREEZE_PATH, SCIENCE_PAYLOAD, m7_science_payload(),
        manifest, LEDGER_RECONCILIATION.to_dict('records'))
    (OUTPUT_DIR/'protocol_freeze.json').write_text(json.dumps(frozen, indent=2)+'\n', encoding='utf-8')
    print('Reviewed validation freeze saved:', FREEZE_PATH)
if RUN_STAGE == 'final':
    shutil.copy2(FREEZE_PATH, OUTPUT_DIR/'protocol_freeze.json')
    shutil.copy2(FINAL_ACCESS_PATH, OUTPUT_DIR/'final_access.json')
SAVE_DIR = SAVE_BASE/RUN_ID
if SAVE_DIR.exists():
    previous = json.loads((SAVE_DIR/'run_manifest.json').read_text())
    if previous['protocol_sha256']!=PROTOCOL_SHA256:
        raise RuntimeError('Existing run folder has a different protocol; preserve it')
shutil.copytree(OUTPUT_DIR,SAVE_DIR,dirs_exist_ok=True)
zip_path = SAVE_BASE/(RUN_ID+'_research_outputs.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as package:
    for file in sorted(SAVE_DIR.iterdir()): package.write(file,arcname=file.name)
print('Saved research outputs:',SAVE_DIR)
print('Supporting ZIP:',zip_path)
print('Final test evaluated:', RUN_STAGE == 'final')


## 13a. Publish a reviewed run (optional, last)

Every run above is saved privately: in `SAVE_FOLDER` on Drive, or in `data/m7_runs/` locally, which git ignores. Failed attempts, test runs and the freeze and access records therefore never reach the public repository by accident.

The next cell lists the saved runs and normally does nothing else. When the group is satisfied with a run, tick `PUBLISH_RUN`, put that run's ID in `PUBLISH_RUN_ID` (or leave it empty for this session's run) and rerun **only that cell**. It checks that the run is complete, contains no prices or trade records and has consistent stage flags, then writes `results/m7_<stage>_<YYYYMMDD>/` with the exports byte for byte, a README, the recorded package versions and `SHA256SUMS.txt`. An existing folder is never overwritten. In Colab, where there is no repository checkout, it prepares the same folder and a ZIP under `SAVE_FOLDER/ready_to_publish/` to download and unzip into `results/`. Publishing copies files only; the publish controls are excluded from the code fingerprint, like the folder paths.


In [ ]:
PUBLISH_RUN = False # @param {type:"boolean"}
PUBLISH_RUN_ID = "" # @param {type:"string"}
# Leave PUBLISH_RUN unticked on every normal run: results stay in the private save folder.
# After the group has reviewed a saved run, tick it, optionally name an earlier RUN_ID, and rerun this cell alone.
# After a runtime restart, run the first code cell, then this one.
# Publishing copies files; it never recomputes, edits or deletes a run.
SAVED_RUNS = []
for manifest_path in sorted(SAVE_BASE.glob('*/run_manifest.json')):
    record = json.loads(manifest_path.read_text(encoding='utf-8'))
    SAVED_RUNS.append(dict(run_id=record.get('run_id'), stage=record.get('stage'),
                           completed_utc=str(record.get('completed_utc', ''))[:19],
                           final_test_evaluated=record.get('final_test_evaluated'),
                           files=sum(1 for p in manifest_path.parent.iterdir() if p.is_file())))
display(Markdown(f'**Saved runs in** `{SAVE_BASE}` (private, not tracked by git)'))
display(pd.DataFrame(SAVED_RUNS) if SAVED_RUNS else Markdown('_No completed runs saved yet._'))

if not PUBLISH_RUN:
    latest = SAVE_BASE / RUN_ID
    print('Not published.', f'This session\'s run is saved privately in {latest}.' if latest.is_dir()
          else 'No run has completed in this session yet.')
    print('When the group is satisfied with a run, tick PUBLISH_RUN (and set PUBLISH_RUN_ID for an earlier run) and rerun this cell alone.')
else:
    publish_id = PUBLISH_RUN_ID.strip() or RUN_ID
    source_dir = SAVE_BASE / publish_id
    if not (source_dir / 'run_manifest.json').is_file():
        raise FileNotFoundError(f'No completed run {publish_id!r} in {SAVE_BASE}. Choose a run_id from the table above.')
    files = {p.name: p.read_bytes() for p in sorted(source_dir.iterdir()) if p.is_file()}
    record = json.loads(files['run_manifest.json'])
    # Quality gates: a complete aggregate export with consistent stage flags, and nothing private.
    required = ['aligned_sensitivity.csv', 'bootstrap_reliability.csv', 'calendar_split.csv', 'coverage_daily.csv',
                'coverage_monthly.csv', 'execution_sensitivity.csv', 'exit_ambiguity.csv', 'expected_shortfall.csv',
                'expected_shortfall_sensitivity.csv', 'funded_drawdown.csv', 'gross_net_returns.csv', 'intervals.csv',
                'ledger_reconciliation.csv', 'paired_effects.csv', 'policy_means.csv', 'sample_accounting.csv',
                'status_counts.csv', 'tail_dates.csv', 'conclusions.md', 'input_audit.json',
                'research_report.html', 'run_manifest.json']
    private_columns = {'open', 'high', 'low', 'close', 'volume', 'price', 'entry_price', 'exit_price', 'stop_price',
                       'target_price', 'net_pnl', 'episode_id', 'signal_at', 'entry_at', 'exit_at'}
    problems = [f'missing export {n}' for n in required if n not in files]
    for name, data in files.items():
        suffix = Path(name).suffix.lower()
        if suffix not in {'.csv', '.json', '.md', '.html', '.png'}:
            problems.append(f'{name}: {suffix or "no extension"} is not an aggregate export')
        if suffix == '.csv':
            header = next(csv.reader(io.StringIO(data.decode('utf-8-sig', errors='replace'))), [])
            leaked = sorted({c.strip().lower() for c in header} & private_columns)
            if leaked:
                problems.append(f'{name} has price or trade-level columns: {", ".join(leaked)}')
    if record.get('stage') == 'final':
        if record.get('final_test_evaluated') is not True or not record.get('final_access_record') or \
           not {'protocol_freeze.json', 'final_access.json'} <= set(files):
            problems.append('a final run needs final_test_evaluated true, its access record, protocol_freeze.json and final_access.json')
    elif record.get('stage') == 'validation':
        if record.get('final_test_evaluated') is not False or 'final_access.json' in files:
            problems.append('a validation run must have final_test_evaluated false and no final_access.json')
    else:
        problems.append(f'stage {record.get("stage")!r}: only validation and final runs are published')
    if problems:
        raise RuntimeError(f'Run {publish_id} cannot be published:\n  - ' + '\n  - '.join(problems))

    completed = datetime.fromisoformat(str(record['completed_utc']).replace('Z', '+00:00'))
    folder_name = f"m7_{record['stage']}_{completed:%Y%m%d}"
    in_repository = REPO_ROOT is not None and (REPO_ROOT / 'results').is_dir()
    publish_root = REPO_ROOT / 'results' if in_repository else SAVE_BASE / 'ready_to_publish'
    base_name = f"m7_{record['stage']}_{completed:%Y%m%d}"
    for suffix in ['', *'bcdefghij']:  # published folders are never overwritten
        folder_name = base_name + suffix
        existing = publish_root / folder_name / 'run_manifest.json'
        if not existing.parent.exists():
            break
        if existing.is_file() and json.loads(existing.read_text(encoding='utf-8')).get('run_id') == publish_id:
            raise RuntimeError(f'Run {publish_id} is already published in {existing.parent}')
    else:
        raise RuntimeError(f'Too many published folders named {base_name}*')
    target = publish_root / folder_name

    period = {'validation': '18 October 2022 to 25 May 2023', 'final': '26 May 2023 to 29 December 2023'}[record['stage']]
    python_version = str(record.get('python', '')).split()[0] or 'not recorded'
    added = {
        '.gitattributes': '# Preserve exact exported bytes and checksum reproducibility on every platform.\n* -text\n',
        'runtime-requirements.txt': '\n'.join(
            [f"# Exact packages recorded for {record['stage']} run {publish_id}",
             f"# Python {python_version}; {record.get('platform', 'platform not recorded')}; archive tools are system dependencies."]
            + [f'{k}=={v}' if v != 'not installed' else f'# {k} not installed'
               for k, v in record.get('packages', {}).items()]) + '\n',
        'README.md': '\n'.join([
            f"# M7 {record['stage']} run, {completed.day} {completed:%B %Y}", '',
            '**Path-Robust Risk Controls for Intraday Mean Reversion in NSE Equities**', '',
            'Uchenna Splendor Ejike, Israel Olanrewaju Odeajo and Goodness Chibueze Kalu · Group 17838', '',
            f"Run `{publish_id}` finished at {completed:%H:%M:%S} UTC on {completed.day} {completed:%B %Y}. "
            f"It is the {record['stage']} stage, covering {period}. "
            f"`final_test_evaluated` is **{str(record['final_test_evaluated']).lower()}**.", '',
            '| Record | Value |', '| --- | --- |',
            f"| Protocol SHA-256 | `{record['protocol_sha256']}` |",
            f"| Notebook code SHA-256 | `{record['notebook_code_sha256']}` |",
            f"| Declared source | `{record.get('declared_source_id', 'not recorded')}` |",
            f"| Source files read | {record.get('source_files', 'not recorded')} (raw inputs re-verified: {record.get('raw_inputs_reverified')}) |",
            f"| Python | {python_version} |", '',
            'Read `gross_net_returns.csv` first for gross returns, assumed fees and net returns, then `paired_effects.csv` '
            'and `intervals.csv` for the exit comparisons, and `expected_shortfall.csv` and `funded_drawdown.csv` for risk. '
            '`conclusions.md` is the notebook\'s own interpretation.' + (
                ' `protocol_freeze.json` is the validation freeze this run was checked against, and `final_access.json` '
                'records the one approved final access.' if record['stage'] == 'final' else
                ' `protocol_freeze.json` is the freeze made from this run after review; the final run is checked against it.'
                if 'protocol_freeze.json' in files else ''), '',
            f'The {len(files)} exports are copied byte for byte from the notebook save folder. `runtime-requirements.txt` '
            'lists the package versions the run recorded, and `SHA256SUMS.txt` covers every other file here. '
            'Raw prices, private ledgers and the executed notebook are not included.']) + '\n',
    }
    contents = {**files, **{k: v.encode('utf-8') for k, v in added.items()}}
    contents['SHA256SUMS.txt'] = ''.join(f'{hashlib.sha256(contents[n]).hexdigest()}  {n}\n'
                                         for n in sorted(contents, key=str.lower)).encode('utf-8')
    target.mkdir(parents=True)
    for name, data in contents.items():
        (target / name).write_bytes(data)
    for line in (target / 'SHA256SUMS.txt').read_text(encoding='utf-8').splitlines():
        digest, name = line.split('  ', 1)
        if hashlib.sha256((target / name).read_bytes()).hexdigest() != digest:
            raise RuntimeError(f'Checksum mismatch after writing {name}')
    print(f'Published run {publish_id} ({record["stage"]}) to {target}: {len(contents)} files, checksums verified.')
    if in_repository:
        print('Review README.md there, then commit it:')
        print(f'  git add results/{folder_name}')
        print(f'  git commit -m "Publish M7 {record["stage"]} run {publish_id}"')
    else:
        shutil.make_archive(str(target), 'zip', root_dir=target)
        print('No repository checkout here, so the folder was prepared in the save folder instead.')
        print(f'Download {target}.zip, unzip it into the repository results/ folder as results/{folder_name}/, and commit it.')


## 14. Contribution and remaining limits

This experiment compares three exits on the same entries, position sizes and expiry times. Its contribution is the measured net-return and risk trade-off under the specified data and execution rules. No positive result is required for completion.

The final report leads with the cost finding, reports the final partition separately when approved and completed, and places detailed qualifications in Limitations. The original topic, stock universe, signal, barriers and calendar boundaries stay fixed. The source notebook, saved aggregates and run records allow the reported calculation to be checked.
